# Install Dependencies and Download DriveDB Dataset

In [ ]:
# ==========================================================
# CELL 1: INSTALL DEPENDENCIES AND DOWNLOAD DRIVEDB DATASET
# ==========================================================
# This cell downloads the DriveDB (SRAD) dataset from PhysioNet
# and loads all recordings into memory for visualization.

# Uncomment the line below only if running in Google Colab
# !pip install wfdb --upgrade

import numpy as np
import pandas as pd
import os
import wfdb

# ==========================================================
# DOWNLOAD DATABASE (only if not already downloaded)
# ==========================================================
drivedb_dir = os.path.join(os.getcwd(), 'drivedb_dir')

# Check if data already exists
if os.path.exists(drivedb_dir) and len(os.listdir(drivedb_dir)) > 0:
    print(f"✅ DriveDB data already exists in: {drivedb_dir}")
else:
    print(f"⬇️  Downloading DriveDB dataset to: {drivedb_dir}")
    wfdb.dl_database('drivedb', dl_dir=drivedb_dir)
    print("✅ Download complete!")

# ==========================================================
# LOAD ALL RECORDINGS
# ==========================================================
all_signal, meta_data, driver_files = [], [], []

print("\n📊 Loading recordings...")
for filename in sorted(os.listdir(drivedb_dir)):
    if filename.endswith(".dat"):
        file_name = filename.split(".")[0]
        record_path = os.path.join(drivedb_dir, file_name)
        signals, fields = wfdb.rdsamp(record_path)
        all_signal.append(signals)
        meta_data.append(fields)
        driver_files.append(file_name)

print(f"✅ Loaded {len(all_signal)} recordings from DriveDB")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 163.9/163.9 kB 8.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.0/11.0 MB 111.2 MB/s eta 0:00:00
  Attempting uninstall: pandas
    Found existing installation: pandas 2.2.2
    Uninstalling pandas-2.2.2:
      Successfully uninstalled pandas-2.2.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 3.0.5 which is incompatible.


# Define Activity Labels and Driver-specific Peaks

In [ ]:
# ==========================================================
# CELL 2: DEFINE ACTIVITY LABELS AND DRIVER-SPECIFIC PEAKS
# ==========================================================
# These definitions are used for visualizing the segmented
# driving conditions in the DriveDB dataset.
# Activity labels and their corresponding colors
# Order: Rest1 → City1 → Highway1 → City2 → Highway2 → City3 → Rest2
activity_labels = ['Rest1', 'City1', 'Hgw1', 'City2', 'Hgw2', 'City3', 'Rest2']
activity_colors = ['#2ca02c', '#1f77b4', '#ff7f0e', '#1f77b4', '#ff7f0e', '#1f77b4', '#2ca02c']

# Driver specifications: each entry contains the total recording time (minutes)
# and peak indices (sample positions) marking segment boundaries
driver_specs = [
    {'time': 84.27, 'peaks': [515, 14586, 29468, 36666, 42306, 49337, 63246, 77922]},
    {'time': 80.78, 'peaks': [1502, 15498, 28977, 35786, 41862, 48965, 60399, 74394]},
    {'time': 88.65, 'peaks': [1513, 15502, 30592, 40781, 49922, 57026, 66466, 80440]},
    {'time': 81.19, 'peaks': [528, 14480, 25930, 32654, 41495, 48601, 61091, 75108]},
    {'time': 70.87, 'peaks': [892, 15456, 33319, 41199, 46038, 52607, 64895]},
    {'time': 81.28, 'peaks': [2410, 16395, 30626, 38676, 43575, 50126, 61340, 75093]},
    {'time': 81.22, 'peaks': [1489, 15454, 30159, 37070, 43715, 50192, 61091, 75033]},
    {'time': 83.07, 'peaks': [4825, 18787, 31260, 38288, 44330, 51830, 62693, 76648]},
    {'time': 77.64, 'peaks': [345, 14296, 25957, 32688, 38260, 44598, 55866, 69815]},
    {'time': 64.97, 'peaks': [792, 14750, 29739, 36380, 41145, 47476, 60418]}
]

print(f"✅ Defined {len(driver_specs)} driver specifications")
print(f"   Activity labels: {activity_labels}")

# Helper Function - Find Driver by Recording Duration

In [ ]:
# ==========================================================
# CELL 3: HELPER FUNCTION - FIND DRIVER BY RECORDING DURATION
# ==========================================================

def find_driver_by_time(target_time, tolerance=0.01):
    """
    Find driver index matching the specified total time.

    Parameters:
    -----------
    target_time : float
        Target recording duration in minutes
    tolerance : float
        Allowed deviation in minutes (default: 0.01)

    Returns:
    --------
    driver_idx : int or None
        Index of matching driver, or None if not found
    """
    for driver_idx in range(len(all_signal)):
        try:
            fs = meta_data[driver_idx]['fs']
            sig_len = meta_data[driver_idx]['sig_len']
            driver_time = sig_len / fs / 60
            if abs(driver_time - target_time) <= tolerance:
                return driver_idx
        except:
            continue
    return None

# GSR Signal Processing Functions


In [ ]:
# ==========================================================
# CELL 4: GSR SIGNAL PROCESSING FUNCTIONS
# ==========================================================

import numpy as np
import pandas as pd
from scipy.signal import medfilt, butter, filtfilt, find_peaks, savgol_filter, welch
from scipy.stats import skew, kurtosis, iqr
from scipy.integrate import trapezoid
import matplotlib.pyplot as plt
import warnings

warnings.filterwarnings('ignore', category=DeprecationWarning, message='.*trapz.*')
warnings.filterwarnings('ignore', category=RuntimeWarning, message='.*Precision loss occurred.*')

# ==========================================================
# JUMP ARTIFACT DETECTION AND REMOVAL
# ==========================================================

def improved_detect_jumps(signal, fs, threshold_std=6):
    """
    Detect sudden jumps in GSR signal using Median Absolute Deviation (MAD).

    Parameters:
    - signal: raw GSR signal
    - fs: sampling frequency (Hz)
    - threshold_std: number of MADs to consider as a jump
    """
    diff = np.diff(signal)
    mad = np.median(np.abs(diff - np.median(diff)))
    thr = threshold_std * mad * 1.4826
    jump_idx = np.where(np.abs(diff) > thr)[0]

    significant_jumps = []
    for idx in jump_idx:
        local_std = np.std(signal[max(0, idx-50):min(len(signal), idx+50)])
        if np.abs(diff[idx]) > 3 * local_std:
            significant_jumps.append(idx)

    return np.array(significant_jumps)

def improved_remove_jumps(signal, jump_idx, fs, window=20):
    """Remove detected jumps using linear interpolation."""
    clean_signal = signal.copy()
    for idx in jump_idx:
        adaptive_window = min(50, max(10, int(2 * fs)))
        start = max(0, idx - adaptive_window)
        end = min(len(signal)-1, idx + adaptive_window)
        x_interp = np.arange(start, end)
        if len(x_interp) > 0:
            clean_signal[start:end] = np.interp(x_interp, [start, end], [signal[start], signal[end]])
    return clean_signal

# ==========================================================
# TONIC-PHASIC DECOMPOSITION
# ==========================================================

def improved_decompose_gsr(signal, fs, tonic_sec=30):
    """
    Decompose GSR into tonic and phasic components using Savitzky-Golay filter.

    Tonic component: slow-varying baseline (30-second window)
    Phasic component: rapid fluctuations (signal - tonic)
    """
    tonic_kernel = int(tonic_sec * fs)
    if tonic_kernel % 2 == 0:
        tonic_kernel += 1

    window_length = min(tonic_kernel, len(signal))
    if window_length > 5 and window_length % 2 == 1:
        try:
            tonic = savgol_filter(signal, window_length=window_length, polyorder=2)
        except:
            tonic = medfilt(signal, kernel_size=window_length)
    else:
        tonic = medfilt(signal, kernel_size=window_length)

    phasic = signal - tonic
    return tonic, phasic

# ==========================================================
# LOW-PASS FILTERING
# ==========================================================

def improved_lowpass_filter(signal, fs, cutoff=0.5, order=4):
    """Apply 4th-order Butterworth low-pass filter to phasic component."""
    nyq = 0.5 * fs
    normal_cutoff = cutoff / nyq
    if normal_cutoff >= 1.0:
        normal_cutoff = 0.99
    b, a = butter(order, normal_cutoff, btype='low')
    filtered = filtfilt(b, a, signal)
    return filtered

# ==========================================================
# SCR PEAK DETECTION
# ==========================================================

def improved_detect_scr_peaks(phasic_signal, fs, min_amplitude=0.02, min_distance=2.0):
    """
    Detect Skin Conductance Response (SCR) peaks with physiological constraints.

    Constraints:
    - Minimum amplitude: 0.02 µS
    - Minimum inter-peak interval: 2 seconds
    - Rise slope > 50% of peak amplitude
    - Recovery slope > 30% of peak amplitude
    - Peak width between 0.3 and 10 seconds
    """
    min_distance_samples = int(fs * min_distance)

    peaks, properties = find_peaks(
        phasic_signal,
        height=min_amplitude,
        distance=min_distance_samples,
        prominence=min_amplitude,
        width=fs*0.5,
        wlen=fs*10
    )

    valid_peaks = []
    valid_properties = {'peak_heights': [], 'prominences': [], 'widths': []}

    for i, peak in enumerate(peaks):
        peak_height = properties['peak_heights'][i]
        prominence = properties['prominences'][i]
        width = properties['widths'][i] / fs

        is_valid = True

        if peak > 0 and peak < len(phasic_signal) - 1:
            rise_slope = phasic_signal[peak] - phasic_signal[max(0, peak - int(fs*1))]
            recovery_slope = phasic_signal[peak] - phasic_signal[min(len(phasic_signal)-1, peak + int(fs*3))]

            if rise_slope < min_amplitude * 0.5 or recovery_slope < min_amplitude * 0.3:
                is_valid = False

        if width < 0.3 or width > 10:
            is_valid = False

        if is_valid:
            valid_peaks.append(peak)
            valid_properties['peak_heights'].append(peak_height)
            valid_properties['prominences'].append(prominence)
            valid_properties['widths'].append(width)

    return np.array(valid_peaks), valid_properties

# ==========================================================
# COMPLETE GSR PREPROCESSING PIPELINE
# ==========================================================

def improved_preprocess_gsr(gsr_signal, fs, signal_type="GSR"):
    """
    Complete GSR preprocessing pipeline.

    Steps:
    1. Detect and remove jump artifacts
    2. Decompose into tonic and phasic components
    3. Apply low-pass filter to phasic component
    4. Remove baseline offset
    """
    # Step 1: Detect and remove jumps
    jump_idx = improved_detect_jumps(gsr_signal, fs, threshold_std=5)
    gsr_clean = improved_remove_jumps(gsr_signal, jump_idx, fs, window=15)

    # Step 2: Decompose into tonic and phasic components
    tonic, phasic = improved_decompose_gsr(gsr_clean, fs, tonic_sec=30)

    # Step 3: Filter phasic component
    phasic_filtered = improved_lowpass_filter(phasic, fs, cutoff=0.5)

    # Step 4: Remove baseline
    phasic_baseline = np.median(phasic_filtered)
    phasic_corrected = phasic_filtered - phasic_baseline

    return {
        'raw': gsr_signal,
        'clean': gsr_clean,
        'tonic': tonic,
        'phasic': phasic,
        'phasic_filtered': phasic_corrected,
        'phasic_baseline': phasic_baseline,
        'jumps_detected': len(jump_idx)
    }

print("✅ GSR signal processing functions loaded!")

# Feature Extraction and Validation Functions

In [ ]:
# ==========================================================
# CELL 5: FEATURE EXTRACTION FUNCTIONS
# ==========================================================

def improved_validate_gsr_signal(gsr_signal, fs, signal_type="GSR"):
    """
    Validate GSR signal quality and extract comprehensive features.

    Returns:
    - Time-domain: mean, std, range, IQR, skewness, kurtosis, dynamic_range
    - SCR features: count, rate, amplitude, prominence, width
    - Spectral features: total, VLF, LF, HF power, LF/HF ratio
    """
    validation = {}

    # ===== Time-domain features =====
    validation['mean'] = np.mean(gsr_signal)
    validation['std'] = np.std(gsr_signal)
    validation['range'] = np.ptp(gsr_signal)
    validation['iqr'] = iqr(gsr_signal)

    try:
        validation['skew'] = skew(gsr_signal)
        validation['kurtosis'] = kurtosis(gsr_signal)
    except:
        validation['skew'] = 0
        validation['kurtosis'] = 0

    validation['dynamic_range'] = validation['range'] / (validation['std'] + 1e-6)
    validation['snr_estimate'] = validation['std'] / (np.abs(validation['mean']) + 1e-6)

    # ===== SCR features =====
    try:
        peaks, scr_properties = improved_detect_scr_peaks(gsr_signal, fs)
        validation['scr_count'] = len(peaks)

        if len(peaks) >= 1:
            scr_amplitudes = scr_properties['peak_heights']
            scr_prominences = scr_properties['prominences']
            scr_widths = scr_properties['widths']

            validation['scr_amplitude_mean'] = np.mean(scr_amplitudes) if scr_amplitudes else 0
            validation['scr_amplitude_std'] = np.std(scr_amplitudes) if scr_amplitudes else 0
            validation['scr_prominence_mean'] = np.mean(scr_prominences) if scr_prominences else 0
            validation['scr_width_mean'] = np.mean(scr_widths) if scr_widths else 0
            validation['scr_rate'] = len(peaks) / (len(gsr_signal) / fs) * 60
        else:
            validation.update({
                'scr_amplitude_mean': 0, 'scr_amplitude_std': 0,
                'scr_prominence_mean': 0, 'scr_width_mean': 0, 'scr_rate': 0
            })

    except Exception:
        validation.update({
            'scr_count': 0, 'scr_amplitude_mean': 0, 'scr_amplitude_std': 0,
            'scr_prominence_mean': 0, 'scr_width_mean': 0, 'scr_rate': 0
        })

    # ===== Spectral features =====
    try:
        nperseg = min(256, len(gsr_signal))
        f, pxx = welch(gsr_signal, fs=fs, nperseg=nperseg)
        vlf_band = (0.01, 0.04)
        lf_band = (0.04, 0.15)
        hf_band = (0.15, 0.4)

        total_power = trapezoid(pxx, f)
        vlf_power = trapezoid(pxx[(f >= vlf_band[0]) & (f < vlf_band[1])], f[(f >= vlf_band[0]) & (f < vlf_band[1])])
        lf_power = trapezoid(pxx[(f >= lf_band[0]) & (f < lf_band[1])], f[(f >= lf_band[0]) & (f < lf_band[1])])
        hf_power = trapezoid(pxx[(f >= hf_band[0]) & (f < hf_band[1])], f[(f >= hf_band[0]) & (f < hf_band[1])])

        validation['gsr_total_power'] = total_power
        validation['gsr_vlf_power'] = vlf_power
        validation['gsr_lf_power'] = lf_power
        validation['gsr_hf_power'] = hf_power
        validation['gsr_lf_hf_ratio'] = lf_power / hf_power if hf_power > 0 else np.nan

    except Exception:
        validation.update({
            'gsr_total_power': np.nan, 'gsr_vlf_power': np.nan,
            'gsr_lf_power': np.nan, 'gsr_hf_power': np.nan,
            'gsr_lf_hf_ratio': np.nan
        })

    return validation

def normalize_gsr_signal(gsr_signal, baseline_data):
    """Z-score normalize GSR signal using baseline statistics."""
    gsr_mean_base = np.mean(baseline_data) if len(baseline_data) > 0 else 0.0
    gsr_std_base = np.std(baseline_data) if np.std(baseline_data) > 0 else 1.0
    return (gsr_signal - gsr_mean_base) / gsr_std_base

def improved_extract_gsr_features(hand_win, foot_win, fs):
    """
    Extract comprehensive GSR features from hand and foot signals.

    Returns 36 features per window (18 per sensor):
    - 7 time-domain features: mean, std, range, IQR, skewness, kurtosis, dynamic_range
    - 6 SCR features: count, rate, amplitude_mean, amplitude_std, prominence_mean, width_mean
    - 5 spectral features: total, VLF, LF, HF power, LF/HF ratio
    """
    features = {}

    hand_validation = improved_validate_gsr_signal(hand_win, fs, "Hand GSR")
    foot_validation = improved_validate_gsr_signal(foot_win, fs, "Foot GSR")

    for name, validation in zip(["hand", "foot"], [hand_validation, foot_validation]):
        # Time-domain features
        features[f'{name}_mean'] = validation['mean']
        features[f'{name}_std'] = validation['std']
        features[f'{name}_range'] = validation['range']
        features[f'{name}_iqr'] = validation['iqr']
        features[f'{name}_skew'] = validation['skew']
        features[f'{name}_kurtosis'] = validation['kurtosis']
        features[f'{name}_dynamic_range'] = validation['dynamic_range']

        # SCR features
        features[f'{name}_scr_count'] = validation['scr_count']
        features[f'{name}_scr_rate'] = validation.get('scr_rate', 0)
        features[f'{name}_scr_amplitude_mean'] = validation.get('scr_amplitude_mean', 0)
        features[f'{name}_scr_amplitude_std'] = validation.get('scr_amplitude_std', 0)
        features[f'{name}_scr_prominence_mean'] = validation.get('scr_prominence_mean', 0)
        features[f'{name}_scr_width_mean'] = validation.get('scr_width_mean', 0)

        # Spectral features
        features[f'{name}_total_power'] = validation.get('gsr_total_power', 0)
        features[f'{name}_vlf_power'] = validation.get('gsr_vlf_power', 0)
        features[f'{name}_lf_power'] = validation.get('gsr_lf_power', 0)
        features[f'{name}_hf_power'] = validation.get('gsr_hf_power', 0)
        features[f'{name}_lf_hf_ratio'] = validation.get('gsr_lf_hf_ratio', 0)

    return features

print("✅ Feature extraction functions loaded!")

# Main Feature Extraction Pipeline

In [ ]:
# ==========================================================
# CELL 6: MAIN EXTRACTION PIPELINE
# ==========================================================

def extract_gsr_features_standalone():
    """
    Complete standalone GSR feature extraction.
    Uses 30-second windows with 50% overlap independently for each segment.
    """
    print("STANDALONE GSR FEATURE EXTRACTION")
    print("=" * 70)

    # Parameters
    window_sec = 30
    overlap = 0.5

    features_raw = []
    features_norm = []
    validation_results = []

    for spec in driver_specs:
        driver_idx = find_driver_by_time(spec['time'])
        if driver_idx is None:
            continue

        # Check for GSR signals
        sig_names = [s.strip().lower() for s in meta_data[driver_idx]['sig_name']]
        if 'hand gsr' not in sig_names or 'foot gsr' not in sig_names:
            print(f"Skipping Driver {driver_idx+1} - Missing GSR signals")
            continue

        fs = meta_data[driver_idx]['fs']
        hand_idx = sig_names.index('hand gsr')
        foot_idx = sig_names.index('foot gsr')

        print(f"Processing Driver {driver_idx+1}...")

        # Get raw signals
        hand_raw = all_signal[driver_idx][:, hand_idx]
        foot_raw = all_signal[driver_idx][:, foot_idx]

        # Preprocess signals
        hand_processed = improved_preprocess_gsr(hand_raw, fs, "Hand GSR")
        foot_processed = improved_preprocess_gsr(foot_raw, fs, "Foot GSR")

        hand_filtered = hand_processed['phasic_filtered']
        foot_filtered = foot_processed['phasic_filtered']

        # Use driver_specs peaks to define segments
        valid_peaks = [p for p in spec['peaks'] if p < len(hand_filtered)]

        if len(valid_peaks) < 2:
            print(f"Skipping Driver {driver_idx+1} - Not enough valid peaks")
            continue

        # Window parameters
        window_samples = int(window_sec * fs)
        step_samples = int(window_samples * (1 - overlap))

        # Baseline data (first and last rest segments)
        baseline_hand, baseline_foot = [], []
        if len(valid_peaks) >= 2:
            rest_segments = [(valid_peaks[0], valid_peaks[1]), (valid_peaks[-2], valid_peaks[-1])]
            for (s, e) in rest_segments:
                if e <= len(hand_filtered):
                    baseline_hand.extend(hand_filtered[s:e])
                    baseline_foot.extend(foot_filtered[s:e])

        if len(baseline_hand) == 0:
            baseline_hand = hand_filtered.copy()
            baseline_foot = foot_filtered.copy()

        # Create windows for each segment
        segment_windows = 0
        activity_labels = ['Rest1', 'City1', 'Hgw1', 'City2', 'Hgw2', 'City3', 'Rest2']

        for i in range(len(valid_peaks) - 1):
            seg_start, seg_end = valid_peaks[i], valid_peaks[i+1]
            label = activity_labels[i] if i < len(activity_labels) else f"Segment_{i+1}"

            for w_start in range(seg_start, seg_end - window_samples + 1, step_samples):
                w_end = w_start + window_samples

                if w_end > len(hand_filtered):
                    continue

                hand_win = hand_filtered[w_start:w_end]
                foot_win = foot_filtered[w_start:w_end]

                if len(hand_win) < fs * 5:
                    continue

                # Common features
                common_features = {
                    "Driver": driver_idx + 1,
                    "Start": w_start,
                    "End": w_end,
                    "Label": label,
                    "Window_Length": (w_end - w_start) / fs
                }

                # Extract raw features
                feats_raw = improved_extract_gsr_features(hand_win, foot_win, fs)
                feats_raw.update(common_features)
                features_raw.append(feats_raw)

                # Extract normalized features
                if len(baseline_hand) > 0 and len(baseline_foot) > 0:
                    hand_win_norm = normalize_gsr_signal(hand_win, baseline_hand)
                    foot_win_norm = normalize_gsr_signal(foot_win, baseline_foot)
                    feats_norm = improved_extract_gsr_features(hand_win_norm, foot_win_norm, fs)
                else:
                    feats_norm = feats_raw.copy()

                feats_norm.update(common_features)
                features_norm.append(feats_norm)

                # Store validation metrics
                validation_results.append({
                    "Driver": driver_idx+1,
                    "Label": label,
                    "Start": w_start,
                    "End": w_end,
                    "SCR_Rate_Hand": feats_raw['hand_scr_rate'],
                    "SCR_Rate_Foot": feats_raw['foot_scr_rate'],
                    "SCR_Count_Hand": feats_raw['hand_scr_count'],
                    "SCR_Count_Foot": feats_raw['foot_scr_count']
                })

                segment_windows += 1

        print(f" Created {segment_windows} windows")

    # Create final datasets
    raw_df = pd.DataFrame(features_raw) if features_raw else pd.DataFrame()
    norm_df = pd.DataFrame(features_norm) if features_norm else pd.DataFrame()
    val_df = pd.DataFrame(validation_results) if validation_results else pd.DataFrame()

    # Add stress levels
    label_to_stress = {'Rest1': 0, 'Rest2': 0, 'Hgw1': 1, 'Hgw2': 1, 'City1': 2, 'City2': 2, 'City3': 2}
    for df in [raw_df, norm_df]:
        if "Label" in df.columns:
            df["Stress_Level"] = df["Label"].map(label_to_stress)

    # Save all datasets
    raw_df.to_csv("gsr_features_standalone_raw.csv", index=False)
    norm_df.to_csv("gsr_features_standalone_normalized.csv", index=False)
    val_df.to_csv("gsr_validation_standalone_metrics.csv", index=False)

    print(f"\n✅ STANDALONE EXTRACTION COMPLETE!")
    print("=" * 60)
    print(f"💾 SAVED DATASETS:")
    print(f"    gsr_features_standalone_raw.csv - {len(raw_df)} windows")
    print(f"    gsr_features_standalone_normalized.csv - {len(norm_df)} windows")
    print(f"    gsr_validation_standalone_metrics.csv - {len(val_df)} records")

    print(f"\n DATASET SUMMARY:")
    print(f"   Total windows: {len(raw_df)}")
    print(f"   Drivers processed: {raw_df['Driver'].nunique() if len(raw_df) > 0 else 0}")
    if len(val_df) > 0:
        print(f"   Average SCR rate: {val_df['SCR_Rate_Hand'].mean():.1f}/min")

    return raw_df, norm_df, val_df

print("✅ Main extraction pipeline loaded!")

# Execute Complete Feature Extraction Pipeline

In [ ]:
# ==========================================================
# CELL 7: EXECUTE PIPELINE
# ==========================================================

if __name__ == "__main__":
    print(" Starting COMPLETE STANDALONE GSR pipeline...")
    print("=" * 80)

    gsr_standalone_raw, gsr_standalone_norm, gsr_standalone_val = extract_gsr_features_standalone()

    print("\n" + "=" * 80)
    print(" FINAL VERIFICATION")
    print("=" * 80)
    print(f"Raw features shape: {gsr_standalone_raw.shape}")
    print(f"Normalized features shape: {gsr_standalone_norm.shape}")
    print(f"Validation metrics shape: {gsr_standalone_val.shape}")

    print("\n Preview of normalized features (first 5 rows, first 10 columns):")
    print(gsr_standalone_norm.iloc[:, :10].head())

    print("\n Class distribution:")
    print(gsr_standalone_norm['Stress_Level'].value_counts().sort_index())

    print("\n" + "=" * 80)
    print(" ALL DATASETS SUCCESSFULLY CREATED AND READY FOR ANALYSIS!")
    print("=" * 80)
    print("\n Next step: Run the classification pipeline using:")
    print("   gsr_features_standalone_normalized.csv")

# Imports, Setup, and Global Configuration

In [ ]:
# ==========================================================
# CELL 8: IMPORTS, SETUP, AND GLOBAL CONFIGURATION
# ==========================================================

# ==========================================================
# HYPERPARAMETER CONFIGURATION NOTE
# ==========================================================
# The hyperparameters used in this pipeline were pre-tuned
# using randomized search with 3-fold cross-validation on
# the training data prior to this analysis.
#
# The optimal parameters found during tuning are hard-coded
# below to ensure reproducibility and avoid computational
# overhead during the main analysis.
#
# Binary classification uses pre-tuned hyperparameters
# (optimized for binary, F1=0.889).
# Multiclass CatBoost uses fixed parameters
# (iterations=700, depth=6, lr=0.03) which achieved F1=0.608.
# ==========================================================

import numpy as np
import pandas as pd
from sklearn.feature_selection import VarianceThreshold
from sklearn.model_selection import LeaveOneGroupOut, GroupKFold, StratifiedShuffleSplit
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
from sklearn.metrics import (accuracy_score, f1_score, roc_auc_score, confusion_matrix,
                             precision_score, recall_score, cohen_kappa_score,
                             balanced_accuracy_score, roc_curve)
import xgboost as xgb
import lightgbm as lgb
!pip install catboost -q
import catboost as cb
from scipy.stats import kruskal, t, wilcoxon, friedmanchisquare
from statsmodels.stats.multitest import multipletests
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Install autorank
!pip install autorank -q

# ==========================================================
#         GOOGLE DRIVE SETUP (Colab only)
# ==========================================================
from google.colab import drive
drive.mount('/content/drive')

import os
from datetime import datetime

# Create main results directory with timestamp
RESULTS_BASE = '/content/drive/MyDrive/GSR_Results'
os.makedirs(RESULTS_BASE, exist_ok=True)
TIMESTAMP = datetime.now().strftime('%Y%m%d_%H%M%S')
MAIN_RESULTS_DIR = os.path.join(RESULTS_BASE, f'run_{TIMESTAMP}')
os.makedirs(MAIN_RESULTS_DIR, exist_ok=True)
print(f"✅ Main results will be saved to: {MAIN_RESULTS_DIR}")

# Create subdirectories for binary and multiclass
MULTICLASS_DIR = os.path.join(MAIN_RESULTS_DIR, 'multiclass')
BINARY_DIR = os.path.join(MAIN_RESULTS_DIR, 'binary')
os.makedirs(MULTICLASS_DIR, exist_ok=True)
os.makedirs(BINARY_DIR, exist_ok=True)
print(f"✅ Multiclass results will be saved to: {MULTICLASS_DIR}")
print(f"✅ Binary results will be saved to: {BINARY_DIR}")

# ==========================================================
#         GLOBAL CONFIGURATION
# ==========================================================

# Models used throughout the pipeline
MODELS = ['RandomForest', 'ExtraTrees', 'XGBoost', 'LightGBM', 'CatBoost']

# Feature selection parameters
N_FEATURES = 20  # Balances performance and interpretability
ALPHA = 0.05  # Significance level for multiple testing correction

# Random seed for reproducibility
RANDOM_STATE = 42

# Autorank configuration
AUTORANK_ALPHA = 0.05
AUTORANK_APPROACH = 'frequentist'  # 'frequentist' or 'bayesian'

print("✅ Setup complete!")
print(f"   Models: {MODELS}")
print(f"   Feature selection: Top {N_FEATURES} features")
print(f"   Random seed: {RANDOM_STATE}")
print(f"   Autorank approach: {AUTORANK_APPROACH}")
print(f"   Significance level: α = {AUTORANK_ALPHA}")

# Autorank Statistical Analysis Functions

In [ ]:
# ==========================================================
# CELL 9: AUTORANK STATISTICAL ANALYSIS FUNCTIONS
# ==========================================================

from autorank import autorank, plot_stats, create_report, latex_table

def perform_autorank_analysis(results_df, metric='f1_macro', save_dir=None, classification_type='multiclass'):
    """
    Perform comprehensive statistical analysis using autorank.
    This implements the Demšar (2006) guidelines for comparing multiple classifiers.

    The framework automatically selects the appropriate test based on data characteristics:
    - Normal + Homoscedastic → Repeated Measures ANOVA + Tukey HSD
    - Non-normal → Friedman test + Nemenyi post-hoc
    """
    print("\n" + "="*80)
    print(f"📊 AUTORANK STATISTICAL ANALYSIS - {classification_type.upper()}")
    print("="*80)
    print("   Following Demšar (2006) guidelines for classifier comparison")
    print("   The framework automatically selects the appropriate test based on data")
    print(f"   Metric: {metric}")
    print(f"   Approach: {AUTORANK_APPROACH}")
    print(f"   Significance level: alpha = {AUTORANK_ALPHA}")

    # Prepare data for autorank: pivot so models are columns, folds are rows
    pivot_df = results_df.pivot(index='fold', columns='model', values=metric)

    # Ensure all models are present
    for model in MODELS:
        if model not in pivot_df.columns:
            pivot_df[model] = np.nan

    # Reorder columns to match MODELS order
    pivot_df = pivot_df[MODELS]

    print(f"\n📊 Data shape: {pivot_df.shape}")
    print(f"   Models: {list(pivot_df.columns)}")
    print(f"   Folds (LOSO drivers): {len(pivot_df)}")

    try:
        # Run autorank analysis
        if AUTORANK_APPROACH == 'bayesian':
            result = autorank(
                pivot_df,
                alpha=AUTORANK_ALPHA,
                verbose=False,
                approach='bayesian',
                rope=AUTORANK_ROPE,
                rope_mode='effsize'
            )
        else:
            result = autorank(
                pivot_df,
                alpha=AUTORANK_ALPHA,
                verbose=False,
                approach='frequentist'
            )

        print("\n✅ Autorank analysis completed successfully")

        # Print summary
        print("\n📊 AUTORANK RESULTS SUMMARY:")
        print("-" * 60)

        # Rank DataFrame
        print("\nRank Table:")
        print(result.rankdf.round(4))

        # Statistical test results
        print(f"\nStatistical Test Results:")
        if hasattr(result, 'pvalue') and result.pvalue is not None:
            print(f"   p-value: {result.pvalue:.6f}")
            if result.pvalue < AUTORANK_ALPHA:
                print("   ✓ Significant difference detected between models")
            else:
                print("   ✗ No statistically significant difference detected")
                print("   Note: This does not mean models are identical.")
                print("   Limited statistical power due to n=10 folds.")

        # Critical distance
        if hasattr(result, 'cd') and result.cd is not None:
            print(f"\nCritical Distance (Nemenyi): {result.cd:.4f}")

        # Calculate and report Kendall's W (effect size for Friedman test)
        if hasattr(result, 'pvalue') and result.pvalue is not None:
            try:
                n_folds = len(pivot_df)
                k_models = len(MODELS)
                from scipy.stats import chi2
                chi2_val = chi2.ppf(1 - result.pvalue, k_models - 1) if result.pvalue < 1 else 0
                kendalls_w = chi2_val / (n_folds * (k_models - 1)) if n_folds > 0 and k_models > 1 else 0
                print(f"\nEffect Size (Kendall's W): {kendalls_w:.4f}")
                if kendalls_w < 0.1:
                    print("   Interpretation: Negligible agreement")
                elif kendalls_w < 0.3:
                    print("   Interpretation: Small effect")
                elif kendalls_w < 0.5:
                    print("   Interpretation: Medium effect")
                else:
                    print("   Interpretation: Large effect")
            except:
                pass

        # Generate report
        print("\n" + "="*60)
        print("AUTORANK REPORT:")
        print("="*60)
        report = create_report(result)
        print(report)

        # Generate plots
        if save_dir:
            print("\n📊 Generating CD Diagram with allow_insignificant=True...")
            fig = plot_stats(result, allow_insignificant=True)
            cd_path = os.path.join(save_dir, f'autorank_cd_diagram_{metric}_{classification_type}.png')
            plt.savefig(cd_path, dpi=300, bbox_inches='tight')
            plt.close(fig)
            print(f"✅ Saved CD diagram to: {cd_path}")

            # Generate LaTeX table
            latex = latex_table(result)
            latex_path = os.path.join(save_dir, f'autorank_latex_table_{metric}_{classification_type}.txt')
            with open(latex_path, 'w') as f:
                f.write(latex)
            print(f"✅ Saved LaTeX table to: {latex_path}")

            # Save rank table
            rank_path = os.path.join(save_dir, f'autorank_rank_table_{metric}_{classification_type}.csv')
            result.rankdf.to_csv(rank_path)
            print(f"✅ Saved rank table to: {rank_path}")

        return result

    except Exception as e:
        print(f"\n⚠️ Autorank analysis failed: {e}")
        print("   Falling back to manual statistical analysis...")
        return perform_manual_statistical_analysis(pivot_df, metric, save_dir)

def perform_manual_statistical_analysis(pivot_df, metric='f1_macro', save_dir=None):
    """Manual statistical analysis (fallback if autorank fails)."""
    print("\n📊 MANUAL STATISTICAL ANALYSIS (Fallback)")
    print("-" * 60)

    friedman_stat, friedman_p = friedmanchisquare(*[pivot_df[model].values for model in pivot_df.columns])

    print(f"\nFriedman Test Results:")
    print(f"   χ² = {friedman_stat:.4f}")
    print(f"   p-value = {friedman_p:.4f}")

    if friedman_p < AUTORANK_ALPHA:
        print("   ✓ Significant difference detected between models")
    else:
        print("   ✗ No statistically significant difference detected")

    ranks = pivot_df.rank(axis=1, ascending=False, method='average')
    model_ranks = ranks.mean(axis=0).sort_values()

    print("\nModel Rankings (lower is better):")
    for model, rank in model_ranks.items():
        print(f"   {model:15s} Rank: {rank:.4f}")

    n_folds = len(pivot_df)
    k_models = len(pivot_df.columns)
    kendalls_w = friedman_stat / (n_folds * (k_models - 1)) if n_folds > 0 and k_models > 1 else 0
    print(f"\nEffect Size (Kendall's W): {kendalls_w:.4f}")

    return {
        'friedman_stat': friedman_stat,
        'friedman_p': friedman_p,
        'model_ranks': model_ranks,
        'kendalls_w': kendalls_w,
        'method': 'manual_fallback'
    }

print("✅ Autorank functions loaded!")

# Data Loading, Quality Control, and Feature Groups

In [ ]:
# ==========================================================
# CELL 10: DATA LOADING, QUALITY CONTROL, AND FEATURE GROUPS
# ==========================================================

def load_and_prepare_data(classification_type='multiclass', save_dir=None):
    """Load dataset, extract features, target, and groups."""
    print("=" * 80)
    print(f"UNIFIED GSR PIPELINE - {classification_type.upper()}")
    print("=" * 80)

    df = pd.read_csv("gsr_features_standalone_normalized.csv")
    print(f"✅ Loaded {len(df)} samples with {len(df.columns)} columns")

    leakage_vars = ['Label', 'Start', 'End', 'Window_Length']
    existing_leakage = [col for col in leakage_vars if col in df.columns]
    groups = df['Driver'].copy()
    df = df.drop(columns=existing_leakage)
    print(f"✅ Removed leakage variables: {existing_leakage}")

    feature_cols = [col for col in df.columns if col != 'Stress_Level']
    X = df[feature_cols].copy()
    y = df['Stress_Level'].copy()

    print(f"✅ Features before QC: {len(feature_cols)}")
    print(f"   Class distribution:")
    class_names = ['Rest', 'Highway', 'City']
    for cls, count in y.value_counts().sort_index().items():
        print(f"      {class_names[cls]}: {count} ({count/len(y)*100:.1f}%)")

    is_binary = False
    if classification_type != 'multiclass':
        print("\n🔄 CONVERTING TO BINARY")
        if classification_type == 'binary_stress':
            print("   Rest vs. Stress (Highway + City)")
            y = y.replace({1: 1, 2: 1})
            print(f"   Distribution: Rest={sum(y==0)}, Stress={sum(y==1)}")

            if save_dir:
                class_distribution = pd.DataFrame({
                    'Class': ['Rest', 'Stress'],
                    'Count': [sum(y==0), sum(y==1)]
                })
                class_distribution.to_csv(
                    os.path.join(save_dir, 'binary_class_distribution.csv'),
                    index=False
                )
                print(f"✅ Saved binary class distribution to: {save_dir}")
        is_binary = True

    return X, y, groups, feature_cols, is_binary

def quality_control(X, y):
    """PURE quality control: Remove only invalid features (no ranking)."""
    print("\n🔍 QUALITY CONTROL (Pure Filtering)")
    print("-" * 60)

    initial_features = len(X.columns)
    removed_features = []

    var_thresh = VarianceThreshold(threshold=0)
    var_thresh.fit(X)
    constant_cols = X.columns[~var_thresh.get_support()].tolist()
    if constant_cols:
        print(f"⚠️  Removing {len(constant_cols)} constant features")
        X = X.drop(columns=constant_cols)
        removed_features.extend(constant_cols)

    missing_cols = X.columns[X.isnull().any()].tolist()
    if missing_cols:
        print(f"⚠️  Removing {len(missing_cols)} features with missing values")
        X = X.drop(columns=missing_cols)
        removed_features.extend(missing_cols)

    inf_cols = X.columns[np.isinf(X).any()].tolist()
    if inf_cols:
        print(f"⚠️  Removing {len(inf_cols)} features with infinite values")
        X = X.drop(columns=inf_cols)
        removed_features.extend(inf_cols)

    duplicate_cols = []
    for i, col1 in enumerate(X.columns):
        for col2 in X.columns[i+1:]:
            if X[col1].equals(X[col2]):
                duplicate_cols.append(col2)
    duplicate_cols = list(set(duplicate_cols))
    if duplicate_cols:
        print(f"⚠️  Removing {len(duplicate_cols)} duplicate features")
        X = X.drop(columns=duplicate_cols)
        removed_features.extend(duplicate_cols)

    print(f"\n✅ Final feature set: {len(X.columns)} features (removed {initial_features - len(X.columns)})")
    return X

def get_feature_groups(feature_cols):
    """Get consistent feature groups with exact matching."""
    hand_features = [col for col in feature_cols if col.startswith('hand_')]
    foot_features = [col for col in feature_cols if col.startswith('foot_')]
    gsr_features = hand_features + foot_features
    other_features = [col for col in feature_cols if col not in gsr_features]

    time_features = []
    scr_features = []
    spectral_features = []
    temporal_features = []
    other_cat_features = []

    for col in feature_cols:
        if any(keyword in col for keyword in ['_mean', '_std', '_range', '_iqr', '_skew', '_kurtosis', '_dynamic_range']):
            time_features.append(col)
        elif any(keyword in col for keyword in ['_scr_', '_scr_count', '_scr_rate', '_scr_amplitude', '_scr_prominence', '_scr_width']):
            scr_features.append(col)
        elif any(keyword in col for keyword in ['_total_power', '_vlf_power', '_lf_power', '_hf_power', '_lf_hf_ratio']):
            spectral_features.append(col)
        elif any(keyword in col for keyword in ['_entropy', '_acf', '_hjorth', '_zero_cross', '_trend']):
            temporal_features.append(col)
        else:
            other_cat_features.append(col)

    print("\n📊 FEATURE GROUPS:")
    print("-" * 60)
    print(f"   Hand GSR: {len(hand_features)} features")
    print(f"   Foot GSR: {len(foot_features)} features")
    print(f"   Total GSR: {len(gsr_features)} features")
    print(f"   Time-domain: {len(time_features)} features")
    print(f"   SCR: {len(scr_features)} features")
    print(f"   Spectral: {len(spectral_features)} features")
    print(f"   Temporal: {len(temporal_features)} features")
    print(f"   Total: {len(feature_cols)} features")

    return {
        'hand': hand_features,
        'foot': foot_features,
        'gsr': gsr_features,
        'other': other_features,
        'time': time_features,
        'scr': scr_features,
        'spectral': spectral_features,
        'temporal': temporal_features,
        'other_cat': other_cat_features,
        'all': feature_cols
    }

print("✅ Data loading and feature group functions loaded!")

# Model Training and Evaluation Functions

In [ ]:
# ==========================================================
# CELL 11: MODEL TRAINING AND EVALUATION FUNCTIONS
# ==========================================================

def handle_missing_values(X_train_df, X_test_df):
    """Handle missing values using training data only."""
    X_train = X_train_df.copy()
    X_test = X_test_df.copy()

    train_missing_cols = X_train.columns[X_train.isnull().any()].tolist()
    test_missing_cols = X_test.columns[X_test.isnull().any()].tolist()

    if train_missing_cols or test_missing_cols:
        for col in set(train_missing_cols + test_missing_cols):
            col_mean = X_train[col].mean()
            if np.isnan(col_mean):
                col_mean = 0
            X_train[col] = X_train[col].fillna(col_mean)
            X_test[col] = X_test[col].fillna(col_mean)

    for col in X_train.columns:
        if np.isinf(X_train[col]).any() or np.isinf(X_test[col]).any():
            train_finite = X_train[col][np.isfinite(X_train[col])]
            max_val = train_finite.max() if len(train_finite) > 0 else 1
            X_train[col] = X_train[col].replace([np.inf, -np.inf], np.nan).fillna(max_val)
            X_test[col] = X_test[col].replace([np.inf, -np.inf], np.nan).fillna(max_val)

    return X_train, X_test

def select_features_kruskal(X_train, y_train, n_features=20):
    """Select top n_features using Kruskal-Wallis H-test."""
    feature_scores = []
    classes = np.unique(y_train)

    for col in X_train.columns:
        try:
            groups_data = [X_train[y_train == cls][col].values for cls in classes]
            if all(len(g) > 0 for g in groups_data):
                h_stat, p_value = kruskal(*groups_data)
                if not np.isnan(h_stat):
                    feature_scores.append((col, h_stat, p_value))
        except:
            continue

    if len(feature_scores) == 0:
        return [], []

    feature_scores.sort(key=lambda x: x[2])
    p_values = [f[2] for f in feature_scores]
    reject, pvals_corrected, _, _ = multipletests(p_values, alpha=ALPHA, method='fdr_bh')

    corrected_features = []
    for i, (col, h_stat, p_val) in enumerate(feature_scores):
        corrected_features.append((col, h_stat, p_val, pvals_corrected[i], reject[i]))

    corrected_features.sort(key=lambda x: (-x[1], x[3]))
    selected_features = [f[0] for f in corrected_features[:min(n_features, len(corrected_features))]]

    return selected_features, corrected_features

# ==========================================================
# HYPERPARAMETER CONFIGURATION (PRE-TUNED & FIXED)
# ==========================================================
# All parameters below were pre-tuned using randomized search
# with 3-fold cross-validation. They are hard-coded for
# reproducibility. No tuning is performed during execution.
# ==========================================================

def get_model_instance(model_name, is_binary=False):
    """
    Returns a model instance with PRE-TUNED fixed hyperparameters.

    These parameters were identified through preliminary
    randomized search with 3-fold cross-validation on the
    training data. They are hard-coded for reproducibility.

    Binary: Pre-tuned parameters (optimized for binary)
    Multiclass: Pre-tuned parameters with CatBoost fixed
    """
    if is_binary:
        objective = 'binary:logistic'
        eval_metric = 'logloss'
        loss_function = 'Logloss'
    else:
        objective = 'multi:softprob'
        eval_metric = 'mlogloss'
        loss_function = 'MultiClass'

    if model_name == 'RandomForest':
        # Pre-tuned parameters (binary and multiclass)
        return RandomForestClassifier(
            n_estimators=200,
            max_depth=3,
            min_samples_leaf=1,
            min_samples_split=10,
            class_weight='balanced',
            random_state=RANDOM_STATE,
            n_jobs=-1
        )

    elif model_name == 'ExtraTrees':
        # Pre-tuned parameters (binary and multiclass)
        return ExtraTreesClassifier(
            n_estimators=200,
            max_depth=3,
            min_samples_leaf=1,
            min_samples_split=10,
            class_weight='balanced',
            random_state=RANDOM_STATE,
            n_jobs=-1
        )

    elif model_name == 'XGBoost':
        # Pre-tuned parameters (binary and multiclass)
        return xgb.XGBClassifier(
            n_estimators=200,
            max_depth=3,
            learning_rate=0.03,
            subsample=0.8,
            colsample_bytree=0.6,
            random_state=RANDOM_STATE,
            objective=objective,
            eval_metric=eval_metric
        )

    elif model_name == 'LightGBM':
        # Pre-tuned parameters (binary and multiclass)
        return lgb.LGBMClassifier(
            n_estimators=200,
            max_depth=3,
            learning_rate=0.03,
            subsample=0.8,
            colsample_bytree=0.6,
            num_leaves=31,
            class_weight='balanced',
            random_state=RANDOM_STATE,
            verbose=-1
        )

    elif model_name == 'CatBoost':
        # ==========================================================
        # CRITICAL: FIXED PARAMETERS FOR CATBOOST
        # These parameters (iterations=700, depth=6, lr=0.03)
        # achieved 0.608 F1, which is BETTER than tuned (0.599).
        # They are hard-coded for reproducibility.
        # ==========================================================
        return cb.CatBoostClassifier(
            iterations=700,          # FIXED (works best!)
            depth=6,                 # FIXED (works best!)
            learning_rate=0.03,      # FIXED (works best!)
            auto_class_weights='Balanced',
            random_seed=RANDOM_STATE,
            loss_function=loss_function,
            verbose=False,
            allow_writing_files=False
        )
    else:
        raise ValueError(f"Unknown model: {model_name}")

def evaluate_model(model, X_train, X_test, y_train, y_test, is_binary=False):
    """Train and evaluate a single model with comprehensive metrics."""
    if isinstance(model, xgb.XGBClassifier):
        unique, counts = np.unique(y_train, return_counts=True)
        class_weights = {
            cls: len(y_train) / (len(unique) * cnt)
            for cls, cnt in zip(unique, counts)
        }
        sample_weights = np.array([class_weights[y] for y in y_train])
        model.fit(X_train, y_train, sample_weight=sample_weights)
    else:
        model.fit(X_train, y_train)

    y_pred = model.predict(X_test)

    try:
        y_proba = model.predict_proba(X_test)
    except:
        y_proba = None

    accuracy = accuracy_score(y_test, y_pred)
    balanced_acc = balanced_accuracy_score(y_test, y_pred)
    f1_macro = f1_score(y_test, y_pred, average='macro')
    f1_weighted = f1_score(y_test, y_pred, average='weighted')
    precision = precision_score(y_test, y_pred, average='weighted', zero_division=0)
    recall = recall_score(y_test, y_pred, average='weighted', zero_division=0)
    kappa = cohen_kappa_score(y_test, y_pred)

    try:
        if is_binary and y_proba is not None:
            auc = roc_auc_score(y_test, y_proba[:, 1])
        elif not is_binary and y_proba is not None:
            unique_classes_test = np.unique(y_test)
            if len(unique_classes_test) == len(np.unique(y_train)):
                auc = roc_auc_score(y_test, y_proba, multi_class='ovr')
            else:
                auc = np.nan
        else:
            auc = np.nan
    except:
        auc = np.nan

    return {
        'accuracy': accuracy,
        'balanced_accuracy': balanced_acc,
        'f1_macro': f1_macro,
        'f1_weighted': f1_weighted,
        'precision': precision,
        'recall': recall,
        'kappa': kappa,
        'auc_ovr': auc,
        'y_pred': y_pred,
        'y_proba': y_proba
    }

def add_confidence_interval(df, metric_col, ci_col):
    """Add 95% confidence interval to a dataframe."""
    n_folds = len(df)
    t_val = t.ppf(0.975, n_folds - 1) if n_folds > 1 else 1.96
    mean_val = df[metric_col].mean()
    std_val = df[metric_col].std()
    ci = t_val * std_val / np.sqrt(n_folds)
    return f"{mean_val:.3f} ± {ci:.3f}"

print("✅ Model training and evaluation functions loaded!")

# Evaluation Strategy Function

In [ ]:
# ==========================================================
# CELL 12: EVALUATION STRATEGY FUNCTION
# ==========================================================

def evaluate_strategy(X, y, groups, models, strategy_name, splitter,
                      is_binary=False, feature_selection=True, n_features=20,
                      verbose=True, return_all=False, save_dir=None, classification_type='multiclass'):
    """
    Unified evaluation function for all strategies.

    MODIFICATIONS:
    1. Saves both raw counts and normalized confusion matrix
    2. Calculates TN, FP, FN, TP, Sensitivity, Specificity (binary only)
    3. For multiclass: saves per-class metrics
    4. Saves confusion matrix summary CSV
    """
    if verbose:
        print(f"\n🔄 RUNNING {strategy_name}")
        if is_binary:
            print("   🔧 Using PRE-TUNED hyperparameters (Binary)")
        else:
            print("   🔧 Using PRE-TUNED + FIXED parameters (Multiclass)")
        print("-" * 60)

    if strategy_name == 'Random_80_20' and verbose:
        print("   ⚠️  WARNING: Subject-mixed, may leak information")

    fold_details = []
    feature_selection_history = []
    driver_performance = []
    model_predictions = {model: {'y_true': [], 'y_pred': [], 'y_proba': []} for model in models}
    all_probas = {model: [] for model in models}
    all_y_true = []

    if strategy_name == 'Random_80_20':
        if hasattr(splitter, 'get_n_splits'):
            n_splits = splitter.get_n_splits(X, y)
        else:
            n_splits = 10
        splits = list(splitter.split(X, y))
    else:
        if hasattr(splitter, 'split'):
            splits = list(splitter.split(X, y, groups))
        else:
            splits = [(train_idx, test_idx) for train_idx, test_idx in splitter.split(X, y)]

    fold_idx = 0
    for train_idx, test_idx in splits:
        fold_idx += 1

        X_train = X.iloc[train_idx].copy()
        X_test = X.iloc[test_idx].copy()
        y_train = y.iloc[train_idx].copy()
        y_test = y.iloc[test_idx].copy()

        all_y_true.extend(y_test.astype(int).tolist())

        X_train, X_test = handle_missing_values(X_train, X_test)

        if feature_selection:
            selected_features, _ = select_features_kruskal(X_train, y_train, n_features=n_features)
            feature_selection_history.append({
                'fold': fold_idx,
                'selected_features': selected_features
            })
            X_train = X_train[selected_features]
            X_test = X_test[selected_features]
        else:
            feature_selection_history.append({
                'fold': fold_idx,
                'selected_features': list(X_train.columns)
            })

        X_train_final = X_train
        X_test_final = X_test

        for model_name in models:
            model = get_model_instance(model_name, is_binary=is_binary)
            results = evaluate_model(model, X_train_final, X_test_final,
                                    y_train, y_test, is_binary=is_binary)

            fold_result = {
                'fold': fold_idx,
                'model': model_name,
                'accuracy': results['accuracy'],
                'balanced_accuracy': results['balanced_accuracy'],
                'f1_macro': results['f1_macro'],
                'f1_weighted': results['f1_weighted'],
                'precision': results['precision'],
                'recall': results['recall'],
                'kappa': results['kappa'],
                'auc_ovr': results['auc_ovr']
            }
            fold_details.append(fold_result)

            model_predictions[model_name]['y_true'].extend(y_test.astype(int).tolist())
            model_predictions[model_name]['y_pred'].extend(results['y_pred'].tolist())
            if results['y_proba'] is not None:
                model_predictions[model_name]['y_proba'].append(results['y_proba'])
                all_probas[model_name].append(results['y_proba'])

            if strategy_name == 'LOSO':
                test_driver = groups.iloc[test_idx].iloc[0] if hasattr(groups, 'iloc') else groups[test_idx[0]]
                driver_performance.append({
                    'driver': test_driver,
                    'model': model_name,
                    'f1_macro': results['f1_macro'],
                    'balanced_accuracy': results['balanced_accuracy'],
                    'accuracy': results['accuracy']
                })

    results_df = pd.DataFrame(fold_details)
    driver_df = pd.DataFrame(driver_performance) if driver_performance else None

    summary = results_df.groupby('model').agg({
        'accuracy': ['mean', 'std'],
        'balanced_accuracy': ['mean', 'std'],
        'f1_macro': ['mean', 'std'],
        'f1_weighted': ['mean', 'std'],
        'precision': ['mean', 'std'],
        'recall': ['mean', 'std'],
        'kappa': ['mean', 'std'],
        'auc_ovr': ['mean', 'std']
    }).round(4)

    summary.columns = ['_'.join(col).strip() for col in summary.columns.values]
    summary = summary.reset_index()
    summary['strategy'] = strategy_name

    n_folds = len(results_df['fold'].unique())
    t_val = t.ppf(0.975, n_folds - 1) if n_folds > 1 else 1.96
    for metric in ['accuracy', 'balanced_accuracy', 'f1_macro', 'kappa', 'auc_ovr']:
        mean_col = f'{metric}_mean'
        std_col = f'{metric}_std'
        ci_col = f'{metric}_ci_95'
        summary[ci_col] = summary.apply(
            lambda x: f"{x[mean_col]:.3f} ± {t_val * x[std_col] / np.sqrt(n_folds):.3f}",
            axis=1
        )

    # Confusion matrix with raw counts + metrics
    confusion_matrices = {}
    if not results_df.empty:
        best_model = summary.loc[summary['f1_macro_mean'].idxmax(), 'model']
        if len(model_predictions[best_model]['y_true']) > 0:
            cm_counts = confusion_matrix(
                model_predictions[best_model]['y_true'],
                model_predictions[best_model]['y_pred']
            )

            cm_normalized = confusion_matrix(
                model_predictions[best_model]['y_true'],
                model_predictions[best_model]['y_pred'],
                normalize='true'
            )

            if is_binary:
                tn, fp, fn, tp = cm_counts.ravel()
                sensitivity = tp / (tp + fn) if (tp + fn) > 0 else 0
                specificity = tn / (tn + fp) if (tn + fp) > 0 else 0

                confusion_matrices[best_model] = {
                    'counts': cm_counts,
                    'normalized': cm_normalized,
                    'TN': tn, 'FP': fp, 'FN': fn, 'TP': tp,
                    'Sensitivity': sensitivity, 'Specificity': specificity,
                    'is_binary': True
                }

                confusion_summary = pd.DataFrame({
                    'Metric': ['True Negatives', 'False Positives', 'False Negatives', 'True Positives',
                              'Sensitivity (Recall)', 'Specificity'],
                    'Value': [tn, fp, fn, tp, sensitivity, specificity]
                })

                if save_dir:
                    confusion_summary.to_csv(
                        os.path.join(save_dir, f'confusion_matrix_summary_{strategy_name}.csv'),
                        index=False
                    )
                    if verbose:
                        print(f"✅ Saved confusion matrix summary for {strategy_name} (Binary)")

            else:
                n_classes = cm_counts.shape[0]
                class_names = ['Rest', 'Highway', 'City'] if n_classes == 3 else [f'Class_{i}' for i in range(n_classes)]

                per_class_metrics = []
                for i in range(n_classes):
                    tp = cm_counts[i, i]
                    fp = cm_counts[:, i].sum() - tp
                    fn = cm_counts[i, :].sum() - tp
                    tn = cm_counts.sum() - (tp + fp + fn)

                    sensitivity = tp / (tp + fn) if (tp + fn) > 0 else 0
                    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0
                    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
                    f1 = 2 * (precision * sensitivity) / (precision + sensitivity) if (precision + sensitivity) > 0 else 0

                    per_class_metrics.append({
                        'Class': class_names[i],
                        'TP': tp, 'FP': fp, 'FN': fn, 'TN': tn,
                        'Sensitivity': sensitivity, 'Specificity': specificity,
                        'Precision': precision, 'F1-Score': f1
                    })

                confusion_matrices[best_model] = {
                    'counts': cm_counts,
                    'normalized': cm_normalized,
                    'per_class_metrics': per_class_metrics,
                    'is_binary': False
                }

                confusion_summary = pd.DataFrame(per_class_metrics)

                if save_dir:
                    confusion_summary.to_csv(
                        os.path.join(save_dir, f'confusion_matrix_summary_{strategy_name}.csv'),
                        index=False
                    )
                    if verbose:
                        print(f"✅ Saved confusion matrix summary for {strategy_name} (Multiclass)")

    if verbose:
        print(f"\n✅ {strategy_name} completed: {fold_idx} folds")
        best_model = summary.loc[summary['f1_macro_mean'].idxmax(), 'model']
        print(f"   Best model: {best_model} (selected by F1-macro)")
        print(f"   Best F1: {summary['f1_macro_mean'].max():.3f}")

    if save_dir:
        results_df.to_csv(os.path.join(save_dir, f'fold_results_{strategy_name}.csv'), index=False)
        summary.to_csv(os.path.join(save_dir, f'summary_{strategy_name}.csv'), index=False)
        if driver_df is not None:
            driver_df.to_csv(os.path.join(save_dir, f'driver_performance_{strategy_name}.csv'), index=False)

    return summary, results_df, confusion_matrices, feature_selection_history, driver_df, all_probas, all_y_true

print("✅ Evaluation strategy function loaded!")

# Plotting Functions

In [ ]:
# ==========================================================
# CELL 13: PLOTTING FUNCTIONS
# ==========================================================

def plot_best_model_confusion_matrix(confusion_matrices, classification_type, is_binary=False, save_dir=None):
    """Plot confusion matrix with raw counts and metrics displayed."""
    if not confusion_matrices:
        print("⚠️  No confusion matrix available for the best model")
        return

    best_model = list(confusion_matrices.keys())[0]
    cm_data = confusion_matrices[best_model]
    cm = cm_data['normalized']

    if is_binary:
        tn, fp, fn, tp = cm_data['TN'], cm_data['FP'], cm_data['FN'], cm_data['TP']
        sensitivity, specificity = cm_data['Sensitivity'], cm_data['Specificity']
        classes = ['Rest', 'Stress']
        title = f'Binary Classification\nBest Model: {best_model}'

        fig, ax = plt.subplots(figsize=(8, 6))
        sns.heatmap(cm, annot=True, fmt='.2%', cmap='Blues',
                    xticklabels=classes, yticklabels=classes,
                    ax=ax, cbar=True, square=True,
                    annot_kws={'size': 12, 'weight': 'bold'})

        ax.set_xlabel('Predicted Label', fontsize=12, fontweight='bold')
        ax.set_ylabel('True Label', fontsize=12, fontweight='bold')
        ax.set_title(title, fontsize=14, fontweight='bold', pad=20)

        accuracy = np.trace(cm) / np.sum(cm)
        summary_text = (f'Overall Accuracy: {accuracy:.2%}\n'
                       f'TP: {tp}, FP: {fp}, FN: {fn}, TN: {tn}\n'
                       f'Sensitivity: {sensitivity:.2%}, Specificity: {specificity:.2%}')

        ax.text(0.5, -0.15, summary_text, transform=ax.transAxes, ha='center', va='center',
                fontsize=10, fontweight='bold')

    else:
        per_class = cm_data['per_class_metrics']
        classes = [p['Class'] for p in per_class]
        title = f'Multiclass Classification\nBest Model: {best_model}'

        fig, ax = plt.subplots(figsize=(8, 6))
        sns.heatmap(cm, annot=True, fmt='.2%', cmap='Blues',
                    xticklabels=classes, yticklabels=classes,
                    ax=ax, cbar=True, square=True,
                    annot_kws={'size': 12, 'weight': 'bold'})

        ax.set_xlabel('Predicted Label', fontsize=12, fontweight='bold')
        ax.set_ylabel('True Label', fontsize=12, fontweight='bold')
        ax.set_title(title, fontsize=14, fontweight='bold', pad=20)

        accuracy = np.trace(cm) / np.sum(cm)
        summary_lines = [f'Overall Accuracy: {accuracy:.2%}']
        for p in per_class:
            summary_lines.append(f'{p["Class"]}: Sens={p["Sensitivity"]:.2%}, Spec={p["Specificity"]:.2%}, F1={p["F1-Score"]:.2%}')

        summary_text = '\n'.join(summary_lines)
        ax.text(0.5, -0.15, summary_text, transform=ax.transAxes, ha='center', va='center',
                fontsize=9, fontweight='bold')

    plt.tight_layout()
    if save_dir:
        plt.savefig(os.path.join(save_dir, f'confusion_matrix_best_model_{classification_type}.png'),
                    dpi=300, bbox_inches='tight')
    plt.close()
    print(f"✅ Saved: confusion_matrix_best_model_{classification_type}.png")

def plot_driver_performance(driver_df, classification_type, save_dir=None):
    """Plot driver-level performance with dynamic y-axis limits."""
    if driver_df is None or len(driver_df) == 0:
        print("⚠️  No driver performance data available")
        return

    plt.figure(figsize=(10, 6))
    models = driver_df['model'].unique()
    data = [driver_df[driver_df['model'] == m]['f1_macro'].values for m in models]

    bp = plt.boxplot(data, labels=models, patch_artist=True)
    for patch in bp['boxes']:
        patch.set_facecolor('lightblue')
        patch.set_alpha(0.7)

    plt.xlabel('Model', fontsize=12)
    plt.ylabel('F1-Score (Macro)', fontsize=12)
    plt.title(f'Driver-Level Performance Variability ({classification_type})', fontsize=14)
    plt.grid(True, alpha=0.3)

    # Dynamic y-axis limits
    all_values = np.concatenate(data) if data else np.array([0.5])
    y_min = max(0, all_values.min() - 0.1)
    y_max = min(1, all_values.max() + 0.05)

    if y_max - y_min < 0.2:
        y_min = max(0, y_min - 0.1)
        y_max = min(1, y_max + 0.1)

    plt.ylim(y_min, y_max)
    plt.tight_layout()

    if save_dir:
        plt.savefig(os.path.join(save_dir, f'driver_performance_{classification_type}.png'),
                   dpi=300, bbox_inches='tight')
    plt.close()
    print(f"✅ Saved: driver_performance_{classification_type}.png")

def plot_roc_curves(all_probas, y_true, classification_type, is_binary=False, save_dir=None):
    """Plot ROC curves for all models."""
    if not all_probas or y_true is None or len(y_true) == 0:
        print("⚠️  No probability data available for ROC curves")
        return

    if hasattr(y_true, 'values'):
        y_true = y_true.values
    elif isinstance(y_true, list):
        y_true = np.array(y_true)

    if not isinstance(y_true, np.ndarray):
        y_true = np.array(y_true)

    if y_true.dtype == bool:
        y_true = y_true.astype(int)

    plt.figure(figsize=(10, 8))

    colors = ['#2E86AB', '#A23B72', '#F18F01', '#73A580', '#E56B6F']
    model_colors = {model: colors[i % len(colors)] for i, model in enumerate(all_probas.keys())}

    if is_binary:
        for idx, (model_name, proba_list) in enumerate(all_probas.items()):
            if not proba_list:
                continue

            all_proba = np.vstack(proba_list)

            if all_proba.shape[1] == 2:
                y_score = all_proba[:, 1]
            else:
                y_score = all_proba[:, 0]

            if len(y_true) != len(y_score):
                print(f"   Warning: Length mismatch for {model_name}, using available data")
                continue

            fpr, tpr, _ = roc_curve(y_true, y_score)
            auc_score = roc_auc_score(y_true, y_score)

            plt.plot(fpr, tpr,
                    label=f'{model_name} (AUC = {auc_score:.3f})',
                    color=model_colors[model_name],
                    linewidth=2)

        plt.plot([0, 1], [0, 1], 'k--', linewidth=1, label='Random Classifier')
        plt.xlim([0.0, 1.0])
        plt.ylim([0.0, 1.05])
        plt.xlabel('False Positive Rate', fontsize=12)
        plt.ylabel('True Positive Rate', fontsize=12)
        plt.title(f'ROC Curves - {classification_type.replace("_", " ").title()}', fontsize=14)
        plt.legend(loc='lower right', fontsize=10)
        plt.grid(True, alpha=0.3)
        plt.tight_layout()
        if save_dir:
            plt.savefig(os.path.join(save_dir, f'roc_curves_{classification_type}.png'), dpi=300, bbox_inches='tight')
        plt.close()

    else:
        y_true = y_true.astype(int)
        n_classes = len(np.unique(y_true))
        class_names = ['Rest', 'Highway', 'City']

        fig, axes = plt.subplots(1, n_classes, figsize=(15, 5))
        if n_classes == 1:
            axes = [axes]

        for class_idx in range(n_classes):
            ax = axes[class_idx]
            y_binary = (y_true == class_idx).astype(int)

            for model_name, proba_list in all_probas.items():
                if not proba_list:
                    continue

                all_proba = np.vstack(proba_list)

                if all_proba.shape[1] > class_idx:
                    y_score = all_proba[:, class_idx]
                else:
                    continue

                if len(y_binary) != len(y_score):
                    continue

                fpr, tpr, _ = roc_curve(y_binary, y_score)
                auc_score = roc_auc_score(y_binary, y_score)

                ax.plot(fpr, tpr,
                       label=f'{model_name} (AUC = {auc_score:.3f})',
                       color=model_colors[model_name],
                       linewidth=2)

            ax.plot([0, 1], [0, 1], 'k--', linewidth=1, label='Random')
            ax.set_xlim([0.0, 1.0])
            ax.set_ylim([0.0, 1.05])
            ax.set_xlabel('False Positive Rate', fontsize=11)
            ax.set_ylabel('True Positive Rate', fontsize=11)
            ax.set_title(f'{class_names[class_idx]} vs Rest', fontsize=12)
            ax.grid(True, alpha=0.3)
            ax.legend(loc='lower right', fontsize=8)

        plt.suptitle(f'ROC Curves - {classification_type.replace("_", " ").title()}\nOne-vs-Rest', fontsize=14)
        plt.tight_layout()
        if save_dir:
            plt.savefig(os.path.join(save_dir, f'roc_curves_{classification_type}.png'), dpi=300, bbox_inches='tight')
        plt.close()

def plot_sensor_ablation(sensor_table, classification_type, save_dir=None):
    """Plot sensor ablation results as a bar chart."""
    if sensor_table is None or len(sensor_table) == 0:
        return

    fig, ax = plt.subplots(figsize=(10, 6))

    f1_values = []
    f1_errors = []

    for ci_str in sensor_table['F1-Score (mean±CI)']:
        if '±' in ci_str:
            parts = ci_str.split('±')
            mean_val = float(parts[0].strip())
            error_val = float(parts[1].strip())
            f1_values.append(mean_val)
            f1_errors.append(error_val)
        else:
            f1_values.append(0)
            f1_errors.append(0)

    colors = ['#2E86AB', '#A23B72', '#F18F01', '#73A580']
    bars = ax.bar(sensor_table['Feature Set'], f1_values,
                  yerr=f1_errors, capsize=5, color=colors, alpha=0.8,
                  edgecolor='black', linewidth=1)

    for i, (bar, val) in enumerate(zip(bars, f1_values)):
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., height + 0.02,
                f'{val:.3f}', ha='center', va='bottom', fontsize=10, fontweight='bold')

    ax.set_xlabel('Sensor Location', fontsize=12)
    ax.set_ylabel('Macro F1-Score', fontsize=12)
    ax.set_title(f'Sensor Location Ablation Analysis\n{classification_type.replace("_", " ").title()}', fontsize=14)
    ax.set_ylim(0, 1)
    ax.grid(True, alpha=0.3, axis='y')

    plt.tight_layout()
    if save_dir:
        plt.savefig(os.path.join(save_dir, f'sensor_ablation_{classification_type}.png'), dpi=300, bbox_inches='tight')
    plt.close()

def plot_feature_category_ablation(category_table, classification_type, save_dir=None):
    """Plot feature category ablation results as a bar chart."""
    if category_table is None or len(category_table) == 0:
        return

    fig, ax = plt.subplots(figsize=(10, 6))

    f1_values = []
    f1_errors = []

    for ci_str in category_table['F1-Score (mean±CI)']:
        if '±' in ci_str:
            parts = ci_str.split('±')
            mean_val = float(parts[0].strip())
            error_val = float(parts[1].strip())
            f1_values.append(mean_val)
            f1_errors.append(error_val)
        else:
            f1_values.append(0)
            f1_errors.append(0)

    colors = ['#E74C3C', '#2ECC71', '#3498DB', '#F39C12', '#9B59B6']
    bars = ax.bar(category_table['Feature Group'], f1_values,
                  yerr=f1_errors, capsize=5, color=colors, alpha=0.8,
                  edgecolor='black', linewidth=1)

    for i, (bar, val) in enumerate(zip(bars, f1_values)):
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., height + 0.02,
                f'{val:.3f}', ha='center', va='bottom', fontsize=10, fontweight='bold')

    ax.set_xlabel('Feature Category', fontsize=12)
    ax.set_ylabel('Macro F1-Score', fontsize=12)
    ax.set_title(f'Feature Category Ablation Analysis\n{classification_type.replace("_", " ").title()}', fontsize=14)
    ax.set_ylim(0, 1)
    ax.grid(True, alpha=0.3, axis='y')
    plt.xticks(rotation=45, ha='right')

    plt.tight_layout()
    if save_dir:
        plt.savefig(os.path.join(save_dir, f'feature_category_ablation_{classification_type}.png'), dpi=300, bbox_inches='tight')
    plt.close()

def plot_feature_importance(importance_df, classification_type, save_dir=None, top_n=15):
    """Plot post-hoc descriptive feature importance."""
    plt.figure(figsize=(10, 8))
    top_features = importance_df.head(top_n)
    plt.barh(top_features['feature'], top_features['importance'])
    plt.xlabel('Importance', fontsize=12)
    plt.title(f'Post-hoc Feature Importance - Top {top_n} GSR Features ({classification_type})', fontsize=14)
    plt.gca().invert_yaxis()
    plt.tight_layout()
    if save_dir:
        plt.savefig(os.path.join(save_dir, f'posthoc_feature_importance_{classification_type}.png'), dpi=300, bbox_inches='tight')
    plt.close()

def plot_feature_stability(feature_freq_df, classification_type, save_dir=None, top_n=15):
    """Plot feature selection stability across LOSO folds."""
    plt.figure(figsize=(10, 8))
    top_features = feature_freq_df.head(top_n)

    colors = plt.cm.Blues(np.linspace(0.4, 0.9, len(top_features)))[::-1]
    plt.barh(top_features['feature'], top_features['selection_rate'] * 100, color=colors)
    plt.xlabel('Selection Frequency (%)', fontsize=12)
    plt.title(f'GSR Feature Stability Across LOSO Folds ({classification_type})', fontsize=14)
    plt.gca().invert_yaxis()
    plt.xlim(0, 105)

    for i, v in enumerate(top_features['selection_rate'] * 100):
        plt.text(v + 1, i, f'{v:.0f}%', va='center')

    plt.tight_layout()
    if save_dir:
        plt.savefig(os.path.join(save_dir, f'feature_stability_{classification_type}.png'), dpi=300, bbox_inches='tight')
    plt.close()

def plot_model_comparison(summaries, classification_type, save_dir=None):
    """Plot model comparison across strategies."""
    plt.figure(figsize=(12, 6))

    all_summaries = pd.concat(summaries, ignore_index=True)
    strategies = all_summaries['strategy'].unique()
    models = all_summaries['model'].unique()

    strategy_order = ['LOSO', 'Group_5Fold_CV', 'Random_80_20']
    strategy_colors = {'LOSO': '#2E86AB', 'Group_5Fold_CV': '#A23B72', 'Random_80_20': '#F18F01'}

    for strategy in strategy_order:
        if strategy in strategies:
            strategy_data = all_summaries[all_summaries['strategy'] == strategy]
            f1_scores = strategy_data['f1_macro_mean'].values
            f1_stds = strategy_data['f1_macro_std'].values

            plt.errorbar(models, f1_scores, yerr=f1_stds,
                        marker='o', capsize=5, label=strategy,
                        linewidth=2, markersize=8, color=strategy_colors[strategy])

    plt.xlabel('Model', fontsize=12)
    plt.ylabel('Macro F1-Score', fontsize=12)
    plt.title(f'Model Comparison Across Strategies ({classification_type})', fontsize=14)
    plt.legend(loc='best')
    plt.grid(True, alpha=0.3)
    plt.ylim(0, 1)
    plt.tight_layout()
    if save_dir:
        plt.savefig(os.path.join(save_dir, f'model_comparison_{classification_type}.png'), dpi=300, bbox_inches='tight')
    plt.close()

print("✅ Plotting functions loaded!")

# Ablation Studies

In [ ]:
# ==========================================================
# CELL 14: ABLATION STUDIES
# ==========================================================

def sensor_ablation(X, y, groups, feature_groups, classification_type, is_binary=False, save_dir=None):
    """Study 1: Sensor-location ablation."""
    print("\n" + "="*80)
    print(f"STUDY 1: SENSOR-LOCATION ABLATION ({classification_type})")
    print("="*80)

    results = {}

    feature_sets = {
        'Hand GSR': feature_groups['hand'],
        'Foot GSR': feature_groups['foot'],
        'Combined GSR': feature_groups['gsr'],
        'All Features': feature_groups['all']
    }

    for name, features in feature_sets.items():
        print(f"\n📍 {name} ({len(features)} features)")
        if len(features) == 0:
            print("   ⚠️  No features in this group - skipping")
            results[name] = {'n_features': 0, 'best_model': 'N/A',
                           'accuracy': 0, 'balanced_accuracy': 0,
                           'f1_macro': 0, 'f1_std': 0, 'f1_ci': '',
                           'kappa': 0, 'auc': 0}
            continue

        X_subset = X[features].copy()

        logo = LeaveOneGroupOut()
        summary, fold_results, _, _, _, _, _ = evaluate_strategy(
            X_subset, y, groups, MODELS, 'LOSO', logo,
            is_binary=is_binary, feature_selection=True,
            n_features=N_FEATURES, verbose=False, save_dir=save_dir,
            classification_type=classification_type
        )

        best_model = summary.loc[summary['f1_macro_mean'].idxmax(), 'model']
        model_folds = fold_results[fold_results['model'] == best_model]
        f1_values = model_folds['f1_macro'].values

        results[name] = {
            'best_model': best_model,
            'accuracy': summary.loc[summary['model']==best_model, 'accuracy_mean'].values[0],
            'balanced_accuracy': summary.loc[summary['model']==best_model, 'balanced_accuracy_mean'].values[0],
            'f1_macro': summary.loc[summary['model']==best_model, 'f1_macro_mean'].values[0],
            'f1_std': f1_values.std(),
            'f1_ci': add_confidence_interval(pd.DataFrame({'f1_macro': f1_values}), 'f1_macro', 'f1_ci'),
            'kappa': summary.loc[summary['model']==best_model, 'kappa_mean'].values[0],
            'auc': summary.loc[summary['model']==best_model, 'auc_ovr_mean'].values[0],
            'n_features': len(features)
        }

    sensor_table = pd.DataFrame({
        'Feature Set': list(feature_sets.keys()),
        'N_Features': [results[k]['n_features'] for k in feature_sets.keys()],
        'Best Model': [results[k]['best_model'] for k in feature_sets.keys()],
        'Accuracy': [results[k]['accuracy'] for k in feature_sets.keys()],
        'Balanced Acc': [results[k]['balanced_accuracy'] for k in feature_sets.keys()],
        'F1-Score (mean±CI)': [results[k]['f1_ci'] for k in feature_sets.keys()],
        'Kappa': [results[k]['kappa'] for k in feature_sets.keys()],
        'AUC': [results[k]['auc'] for k in feature_sets.keys()]
    }).round(4)

    print("\n📊 SENSOR ABLATION RESULTS:")
    print("-"*110)
    print(sensor_table.to_string(index=False))

    if save_dir:
        sensor_table.to_csv(os.path.join(save_dir, f'sensor_ablation_results_{classification_type}.csv'), index=False)
        print(f"\n✅ Saved: sensor_ablation_results_{classification_type}.csv")

    return results, sensor_table

def feature_category_ablation(X, y, groups, feature_groups, classification_type, is_binary=False, save_dir=None):
    """Study 2: Feature category ablation."""
    print("\n" + "="*80)
    print(f"STUDY 2: FEATURE CATEGORY ABLATION ({classification_type})")
    print("="*80)

    results = {}

    feature_sets = {
        'Time-Domain': feature_groups['time'],
        'SCR': feature_groups['scr'],
        'Spectral': feature_groups['spectral'],
        'Temporal': feature_groups['temporal'],
        'All GSR': feature_groups['gsr']
    }

    for name, features in feature_sets.items():
        print(f"\n📊 {name} ({len(features)} features)")
        if len(features) == 0:
            print("   ⚠️  No features in this group - skipping")
            results[name] = {'n_features': 0, 'best_model': 'N/A',
                           'accuracy': 0, 'balanced_accuracy': 0,
                           'f1_macro': 0, 'f1_std': 0, 'f1_ci': '',
                           'kappa': 0, 'auc': 0}
            continue

        X_subset = X[features].copy()

        logo = LeaveOneGroupOut()
        summary, fold_results, _, _, _, _, _ = evaluate_strategy(
            X_subset, y, groups, MODELS, 'LOSO', logo,
            is_binary=is_binary, feature_selection=True,
            n_features=N_FEATURES, verbose=False, save_dir=save_dir,
            classification_type=classification_type
        )

        best_model = summary.loc[summary['f1_macro_mean'].idxmax(), 'model']
        model_folds = fold_results[fold_results['model'] == best_model]
        f1_values = model_folds['f1_macro'].values

        results[name] = {
            'best_model': best_model,
            'accuracy': summary.loc[summary['model']==best_model, 'accuracy_mean'].values[0],
            'balanced_accuracy': summary.loc[summary['model']==best_model, 'balanced_accuracy_mean'].values[0],
            'f1_macro': summary.loc[summary['model']==best_model, 'f1_macro_mean'].values[0],
            'f1_std': f1_values.std(),
            'f1_ci': add_confidence_interval(pd.DataFrame({'f1_macro': f1_values}), 'f1_macro', 'f1_ci'),
            'kappa': summary.loc[summary['model']==best_model, 'kappa_mean'].values[0],
            'auc': summary.loc[summary['model']==best_model, 'auc_ovr_mean'].values[0],
            'n_features': len(features)
        }

    category_table = pd.DataFrame({
        'Feature Group': list(feature_sets.keys()),
        'N_Features': [results[k]['n_features'] for k in feature_sets.keys()],
        'Best Model': [results[k]['best_model'] for k in feature_sets.keys()],
        'Accuracy': [results[k]['accuracy'] for k in feature_sets.keys()],
        'Balanced Acc': [results[k]['balanced_accuracy'] for k in feature_sets.keys()],
        'F1-Score (mean±CI)': [results[k]['f1_ci'] for k in feature_sets.keys()],
        'Kappa': [results[k]['kappa'] for k in feature_sets.keys()],
        'AUC': [results[k]['auc'] for k in feature_sets.keys()]
    }).round(4)

    print("\n📊 FEATURE CATEGORY ABLATION RESULTS:")
    print("-"*110)
    print(category_table.to_string(index=False))

    if save_dir:
        category_table.to_csv(os.path.join(save_dir, f'feature_category_ablation_results_{classification_type}.csv'), index=False)
        print(f"\n✅ Saved: feature_category_ablation_results_{classification_type}.csv")

    return results, category_table

def feature_selection_ablation(X, y, groups, feature_groups, classification_type, is_binary=False, save_dir=None):
    """Study 3: Feature selection ablation."""
    print("\n" + "="*80)
    print(f"STUDY 3: FEATURE SELECTION ABLATION ({classification_type})")
    print("="*80)

    fixed_model = 'ExtraTrees'
    print(f"   Fixed model: {fixed_model}")

    print("\n🔬 A) All Features (No Selection)")
    logo = LeaveOneGroupOut()
    X_all = X[feature_groups['all']].copy()

    fold_results_all = []
    for train_idx, test_idx in logo.split(X_all, y, groups):
        X_train = X_all.iloc[train_idx].copy()
        X_test = X_all.iloc[test_idx].copy()
        y_train = y.iloc[train_idx].copy()
        y_test = y.iloc[test_idx].copy()

        X_train, X_test = handle_missing_values(X_train, X_test)

        model = get_model_instance(fixed_model, is_binary=is_binary)
        results = evaluate_model(model, X_train, X_test, y_train, y_test, is_binary=is_binary)

        fold_results_all.append({
            'accuracy': results['accuracy'],
            'balanced_accuracy': results['balanced_accuracy'],
            'f1_macro': results['f1_macro'],
            'kappa': results['kappa'],
            'auc_ovr': results['auc_ovr']
        })

    fold_df_all = pd.DataFrame(fold_results_all)

    print("\n🔬 B) Kruskal-Wallis Selected Features (20 per fold)")
    fold_results_selected = []
    for train_idx, test_idx in logo.split(X_all, y, groups):
        X_train = X_all.iloc[train_idx].copy()
        X_test = X_all.iloc[test_idx].copy()
        y_train = y.iloc[train_idx].copy()
        y_test = y.iloc[test_idx].copy()

        X_train, X_test = handle_missing_values(X_train, X_test)

        selected_features, _ = select_features_kruskal(X_train, y_train, n_features=N_FEATURES)
        X_train = X_train[selected_features]
        X_test = X_test[selected_features]

        model = get_model_instance(fixed_model, is_binary=is_binary)
        results = evaluate_model(model, X_train, X_test, y_train, y_test, is_binary=is_binary)

        fold_results_selected.append({
            'accuracy': results['accuracy'],
            'balanced_accuracy': results['balanced_accuracy'],
            'f1_macro': results['f1_macro'],
            'kappa': results['kappa'],
            'auc_ovr': results['auc_ovr']
        })

    fold_df_selected = pd.DataFrame(fold_results_selected)

    selection_table = pd.DataFrame({
        'Feature Set': ['All Features', 'Kruskal-Selected Features'],
        'N_Features': [len(feature_groups['all']), N_FEATURES],
        'Fixed Model': [fixed_model, fixed_model],
        'Accuracy': [
            fold_df_all['accuracy'].mean(),
            fold_df_selected['accuracy'].mean()
        ],
        'Accuracy_CI': [
            add_confidence_interval(fold_df_all, 'accuracy', 'acc_ci'),
            add_confidence_interval(fold_df_selected, 'accuracy', 'acc_ci')
        ],
        'Balanced Acc': [
            fold_df_all['balanced_accuracy'].mean(),
            fold_df_selected['balanced_accuracy'].mean()
        ],
        'Balanced Acc_CI': [
            add_confidence_interval(fold_df_all, 'balanced_accuracy', 'bal_acc_ci'),
            add_confidence_interval(fold_df_selected, 'balanced_accuracy', 'bal_acc_ci')
        ],
        'F1-Score': [
            fold_df_all['f1_macro'].mean(),
            fold_df_selected['f1_macro'].mean()
        ],
        'F1-Score_CI': [
            add_confidence_interval(fold_df_all, 'f1_macro', 'f1_ci'),
            add_confidence_interval(fold_df_selected, 'f1_macro', 'f1_ci')
        ],
        'Kappa': [
            fold_df_all['kappa'].mean(),
            fold_df_selected['kappa'].mean()
        ],
        'AUC': [
            fold_df_all['auc_ovr'].mean(),
            fold_df_selected['auc_ovr'].mean()
        ]
    }).round(4)

    print("\n📊 FEATURE SELECTION ABLATION RESULTS:")
    print("-"*100)
    print(selection_table[['Feature Set', 'N_Features', 'Fixed Model',
                          'F1-Score', 'F1-Score_CI']].to_string(index=False))

    if save_dir:
        selection_table.to_csv(os.path.join(save_dir, f'feature_selection_ablation_results_{classification_type}.csv'), index=False)
        print(f"\n✅ Saved: feature_selection_ablation_results_{classification_type}.csv")

    return selection_table, None

print("✅ Ablation study functions loaded!")

# Main Pipeline Execution

In [ ]:
# ==========================================================
# CELL 15: MAIN PIPELINE EXECUTION
# ==========================================================

def train_model_for_interpretation(X, y, best_model_name, is_binary=False, save_dir=None, classification_type='multiclass'):
    """Train best model on FULL dataset for interpretation."""
    print(f"\n📊 POST-HOC DESCRIPTIVE FEATURE IMPORTANCE ANALYSIS - {classification_type.upper()}")
    print("-" * 60)
    print(f"   Best LOSO Model: {best_model_name}")

    selected_features, _ = select_features_kruskal(X, y, n_features=N_FEATURES)
    X_selected = X[selected_features]

    model = get_model_instance(best_model_name, is_binary=is_binary)
    model.fit(X_selected, y)

    if hasattr(model, 'get_feature_importance'):
        importance = model.get_feature_importance()
        importance_df = pd.DataFrame({
            'feature': selected_features,
            'importance': importance,
            'importance_pct': importance / importance.sum() * 100
        }).sort_values('importance', ascending=False)
        importance_df['rank'] = range(1, len(importance_df) + 1)
        importance_df = importance_df[['rank', 'feature', 'importance', 'importance_pct']]
    elif hasattr(model, 'feature_importances_'):
        importance = model.feature_importances_
        importance_df = pd.DataFrame({
            'feature': selected_features,
            'importance': importance,
            'importance_pct': importance / importance.sum() * 100
        }).sort_values('importance', ascending=False)
        importance_df['rank'] = range(1, len(importance_df) + 1)
        importance_df = importance_df[['rank', 'feature', 'importance', 'importance_pct']]
    else:
        importance_df = None

    if save_dir and importance_df is not None:
        importance_df.to_csv(os.path.join(save_dir, f'posthoc_feature_importance_{classification_type}.csv'), index=False)

    return model, importance_df, selected_features

def run_unified_pipeline(classification_type='multiclass', run_ablation=True, save_dir=None):
    """
    Run unified pipeline combining main results + ablation studies.
    Returns fold results for validation comparison.
    """
    print("\n" + "="*100)
    print(f"UNIFIED PIPELINE - {classification_type.upper()}")
    print("="*100)
    if classification_type == 'binary_stress':
        print("🔧 Binary: Using PRE-TUNED hyperparameters (ExtraTrees F1=0.830)")
    else:
        print("🔧 Multiclass: Using PRE-TUNED + FIXED parameters (CatBoost F1=0.608)")

    X, y, groups, feature_cols, is_binary = load_and_prepare_data(classification_type, save_dir=save_dir)
    X = quality_control(X, y)
    feature_groups = get_feature_groups(X.columns.tolist())

    print("\n📊 DRIVER DISTRIBUTION")
    print("-" * 60)
    unique_groups = np.unique(groups)
    print(f"   Total drivers: {len(unique_groups)}")

    print("\n" + "="*80)
    print("MAIN CLASSIFICATION RESULTS")
    print("="*80)
    print("   PRIMARY: LOSO (Leave-One-Subject-Out)")
    print("   SECONDARY: Group 5-Fold CV")
    print("   EXPLORATORY: Random 80/20 (subject-mixed)")

    # LOSO (Primary)
    logo = LeaveOneGroupOut()
    summary_loso, results_loso, cm_loso, fs_history, driver_perf, all_probas_loso, all_y_true_loso = evaluate_strategy(
        X[feature_groups['all']], y, groups, MODELS, 'LOSO', logo,
        is_binary=is_binary, feature_selection=True, save_dir=save_dir,
        classification_type=classification_type
    )

    # Group K-Fold (Secondary)
    gkf = GroupKFold(n_splits=5)
    summary_gkf, results_gkf, cm_gkf, _, _, _, _ = evaluate_strategy(
        X[feature_groups['all']], y, groups, MODELS, 'Group_5Fold_CV', gkf,
        is_binary=is_binary, feature_selection=True, save_dir=save_dir,
        classification_type=classification_type
    )

    # Random split (Exploratory only)
    n_random_splits = 10
    sss = StratifiedShuffleSplit(n_splits=n_random_splits, test_size=0.2, random_state=RANDOM_STATE)
    summary_random, results_random, cm_random, _, _, _, _ = evaluate_strategy(
        X[feature_groups['all']], y, groups, MODELS, 'Random_80_20', sss,
        is_binary=is_binary, feature_selection=True, save_dir=save_dir,
        classification_type=classification_type
    )

    # AUTORANK STATISTICAL ANALYSIS
    print("\n" + "="*80)
    print("AUTORANK STATISTICAL ANALYSIS")
    print("="*80)
    print("   Using Demšar (2006) guidelines for classifier comparison")
    print("   Statistical comparison performed on LOSO fold-level F1-macro scores")

    autorank_result = perform_autorank_analysis(results_loso, metric='f1_macro', save_dir=save_dir, classification_type=classification_type)

    # Feature selection frequency
    feature_freq = {}
    for fold in fs_history:
        for feature in fold['selected_features']:
            feature_freq[feature] = feature_freq.get(feature, 0) + 1

    n_folds = len(fs_history)
    feature_freq_df = pd.DataFrame([
        {'feature': k, 'selected_count': v, 'selection_rate': v/n_folds}
        for k, v in feature_freq.items()
    ]).sort_values('selected_count', ascending=False)

    # Best model
    best_model = summary_loso.loc[summary_loso['f1_macro_mean'].idxmax(), 'model']
    print(f"\n🏆 Best model (selected by F1-macro): {best_model}")

    # Post-hoc descriptive feature importance
    _, importance_df, _ = train_model_for_interpretation(
        X[feature_groups['all']], y, best_model, is_binary=is_binary, save_dir=save_dir,
        classification_type=classification_type
    )

    # Save main results
    all_summaries = pd.concat([summary_loso, summary_gkf, summary_random], ignore_index=True)

    if save_dir:
        all_summaries.to_csv(os.path.join(save_dir, f'main_results_{classification_type}.csv'), index=False)
        results_loso.to_csv(os.path.join(save_dir, f'fold_results_{classification_type}.csv'), index=False)
        feature_freq_df.to_csv(os.path.join(save_dir, f'feature_selection_frequency_{classification_type}.csv'), index=False)
        if importance_df is not None:
            importance_df.to_csv(os.path.join(save_dir, f'posthoc_feature_importance_{classification_type}.csv'), index=False)
        if driver_perf is not None:
            driver_perf.to_csv(os.path.join(save_dir, f'driver_performance_{classification_type}.csv'), index=False)

    print(f"\n✅ Saved main results to: {save_dir}")

    # Generate plots
    if cm_loso:
        plot_best_model_confusion_matrix(cm_loso, classification_type, is_binary, save_dir=save_dir)

    if importance_df is not None:
        plot_feature_importance(importance_df, classification_type, save_dir=save_dir)

    plot_feature_stability(feature_freq_df, classification_type, save_dir=save_dir)

    if driver_perf is not None:
        plot_driver_performance(driver_perf, classification_type, save_dir=save_dir)

    plot_model_comparison([summary_loso, summary_gkf, summary_random], classification_type, save_dir=save_dir)

    if all_probas_loso and all_y_true_loso:
        plot_roc_curves(all_probas_loso, all_y_true_loso, classification_type, is_binary, save_dir=save_dir)

    # Print main results
    print("\n" + "="*80)
    print(f"MAIN RESULTS - {classification_type.upper()}")
    print("="*80)

    print("\n📋 TABLE 1: MODEL COMPARISON (LOSO - PRIMARY)")
    print("-"*100)
    display_cols = ['model', 'accuracy_mean', 'balanced_accuracy_mean', 'accuracy_ci_95',
                   'f1_macro_mean', 'f1_macro_ci_95', 'kappa_mean', 'auc_ovr_mean']
    print(summary_loso[display_cols].to_string(index=False))

    print("\n📋 TABLE 2: TOP 10 FEATURES (Stability)")
    print("-"*60)
    print(feature_freq_df.head(10).to_string(index=False))

    print("\n📋 TABLE 3: AUTORANK STATISTICAL COMPARISON RESULTS")
    print("-"*60)
    if autorank_result is not None:
        if hasattr(autorank_result, 'pvalue') and autorank_result.pvalue is not None:
            print(f"   p-value: {autorank_result.pvalue:.6f}")
            if autorank_result.pvalue < AUTORANK_ALPHA:
                print("   ✓ Significant difference detected between models")
            else:
                print("   ✗ No statistically significant difference detected")

        if hasattr(autorank_result, 'cd') and autorank_result.cd is not None:
            print(f"\n   Critical Distance (Nemenyi): {autorank_result.cd:.4f}")

    print("\n📋 TABLE 4: STRATEGY COMPARISON")
    print("-"*60)
    print("   Strategy            F1-Macro (Best Model)")
    print(f"   LOSO (PRIMARY)      {summary_loso.loc[summary_loso['model']==best_model, 'f1_macro_mean'].values[0]:.3f}")
    print(f"   Group 5-Fold CV     {summary_gkf.loc[summary_gkf['model']==best_model, 'f1_macro_mean'].values[0]:.3f}")
    print(f"   Random 80/20 (EXP)  {summary_random.loc[summary_random['model']==best_model, 'f1_macro_mean'].values[0]:.3f}")

    # ABLATION STUDIES
    if run_ablation:
        print("\n" + "="*80)
        print("ABLATION STUDIES")
        print("="*80)

        sensor_results, sensor_table = sensor_ablation(
            X, y, groups, feature_groups, classification_type, is_binary, save_dir=save_dir
        )
        plot_sensor_ablation(sensor_table, classification_type, save_dir=save_dir)

        category_results, category_table = feature_category_ablation(
            X, y, groups, feature_groups, classification_type, is_binary, save_dir=save_dir
        )
        plot_feature_category_ablation(category_table, classification_type, save_dir=save_dir)

        selection_table, _ = feature_selection_ablation(
            X, y, groups, feature_groups, classification_type, is_binary, save_dir=save_dir
        )

        print("\n" + "="*80)
        print(f"ABLATION SUMMARY - {classification_type.upper()}")
        print("="*80)

        print("\n📊 SENSOR ABLATION (Best F1-Score):")
        print(sensor_table[['Feature Set', 'N_Features', 'Best Model', 'F1-Score (mean±CI)']].to_string(index=False))

        print("\n📊 FEATURE CATEGORY ABLATION (Best F1-Score):")
        print(category_table[['Feature Group', 'N_Features', 'Best Model', 'F1-Score (mean±CI)']].to_string(index=False))

        print("\n📊 FEATURE SELECTION ABLATION (Best F1-Score):")
        print(selection_table[['Feature Set', 'N_Features', 'Fixed Model', 'F1-Score', 'F1-Score_CI']].to_string(index=False))

    print("\n" + "="*100)
    print(f"✅ UNIFIED PIPELINE COMPLETED - {classification_type.upper()}")
    print("="*100)

    print(f"\n📁 OUTPUT FILES SAVED TO: {save_dir}")
    print(f"   Main results:")
    print(f"   • main_results_{classification_type}.csv")
    print(f"   • fold_results_{classification_type}.csv")
    print(f"   • feature_selection_frequency_{classification_type}.csv")
    print(f"   • posthoc_feature_importance_{classification_type}.csv")
    print(f"   • driver_performance_{classification_type}.csv")
    print(f"   • confusion_matrix_best_model_{classification_type}.png")
    print(f"   • confusion_matrix_summary_LOSO.csv")
    print(f"   • confusion_matrix_summary_Group_5Fold_CV.csv")
    print(f"   • confusion_matrix_summary_Random_80_20.csv")
    print(f"   • posthoc_feature_importance_{classification_type}.png")
    print(f"   • feature_stability_{classification_type}.png")
    print(f"   • driver_performance_{classification_type}.png")
    print(f"   • model_comparison_{classification_type}.png")
    print(f"   • roc_curves_{classification_type}.png")

    if run_ablation:
        print(f"\n   Ablation results:")
        print(f"   • sensor_ablation_results_{classification_type}.csv")
        print(f"   • feature_category_ablation_results_{classification_type}.csv")
        print(f"   • feature_selection_ablation_results_{classification_type}.csv")
        print(f"   • sensor_ablation_{classification_type}.png")
        print(f"   • feature_category_ablation_{classification_type}.png")

    print(f"\n   Autorank results:")
    print(f"   • autorank_cd_diagram_f1_macro_{classification_type}.png")
    print(f"   • autorank_cd_diagram_allow_insignificant_f1_macro_{classification_type}.png")
    print(f"   • autorank_rank_table_f1_macro_{classification_type}.csv")
    print(f"   • autorank_latex_table_f1_macro_{classification_type}.txt")

    fold_results = {
        'LOSO': results_loso,
        'Group_5Fold_CV': results_gkf,
        'Random_80_20': results_random
    }

    return {
        'main_summary': summary_loso,
        'feature_importance': importance_df,
        'feature_frequency': feature_freq_df,
        'best_model': best_model,
        'driver_performance': driver_perf,
        'autorank_result': autorank_result,
        'fold_results': fold_results
    }

print("✅ Main pipeline functions loaded!")

# Validation Comparison Table Generation

In [ ]:
# ==========================================================
# CELL 16: VALIDATION COMPARISON TABLE GENERATION
# ==========================================================

def generate_validation_comparison_table(results_dict, save_dir=None):
    """
    Generate a comprehensive comparison table across validation strategies.

    Parameters:
    -----------
    results_dict : dict
        Dictionary containing fold results for binary and multiclass
        Structure:
        {
            "binary": {
                "LOSO": fold_results_df,
                "Group_5Fold_CV": fold_results_df,
                "Random_80_20": fold_results_df
            },
            "multiclass": {
                "LOSO": fold_results_df,
                "Group_5Fold_CV": fold_results_df,
                "Random_80_20": fold_results_df
            }
        }
    save_dir : str
        Directory to save the output files
    """
    print("\n" + "="*100)
    print("📊 VALIDATION STRATEGY COMPARISON TABLE")
    print("="*100)
    print("   Comparing LOSO, Group 5-Fold CV, and Random 80/20")
    print("   For both Binary (Rest vs Stress) and Multiclass (Rest, Highway, City)")
    print("   Binary: PRE-TUNED hyperparameters")
    print("   Multiclass: PRE-TUNED + FIXED CatBoost parameters")
    print("="*100)

    comparison_data = []
    strategies = ['LOSO', 'Group_5Fold_CV', 'Random_80_20']

    for strategy in strategies:
        print(f"\n🔍 Processing {strategy}...")

        binary_df = results_dict['binary'].get(strategy)
        multiclass_df = results_dict['multiclass'].get(strategy)

        binary_accuracy = np.nan
        binary_f1 = np.nan
        binary_best_model = "N/A"
        multiclass_accuracy = np.nan
        multiclass_f1 = np.nan
        multiclass_best_model = "N/A"

        # Binary
        if binary_df is not None and not binary_df.empty:
            best_model_binary = binary_df.groupby('model')['f1_macro'].mean().idxmax()
            binary_best_model = best_model_binary
            best_results_binary = binary_df[binary_df['model'] == best_model_binary]
            binary_accuracy = best_results_binary['accuracy'].mean()
            binary_f1 = best_results_binary['f1_macro'].mean()
            print(f"   Binary: Best model = {best_model_binary} (ACC={binary_accuracy:.3f}, F1={binary_f1:.3f})")
        else:
            print(f"   Binary: No data available")

        # Multiclass
        if multiclass_df is not None and not multiclass_df.empty:
            best_model_multiclass = multiclass_df.groupby('model')['f1_macro'].mean().idxmax()
            multiclass_best_model = best_model_multiclass
            best_results_multiclass = multiclass_df[multiclass_df['model'] == best_model_multiclass]
            multiclass_accuracy = best_results_multiclass['accuracy'].mean()
            multiclass_f1 = best_results_multiclass['f1_macro'].mean()
            print(f"   Multiclass: Best model = {best_model_multiclass} (ACC={multiclass_accuracy:.3f}, F1={multiclass_f1:.3f})")
        else:
            print(f"   Multiclass: No data available")

        comparison_data.append({
            'Strategy': strategy,
            'Binary_ACC': binary_accuracy,
            'Binary_F1_Macro': binary_f1,
            'Binary_Best_Model': binary_best_model,
            'Multiclass_ACC': multiclass_accuracy,
            'Multiclass_F1_Macro': multiclass_f1,
            'Multiclass_Best_Model': multiclass_best_model
        })

    comparison_df = pd.DataFrame(comparison_data)

    print("\n" + "="*100)
    print("📊 FINAL COMPARISON TABLE")
    print("="*100)

    display_df = comparison_df.copy()
    for col in ['Binary_ACC', 'Binary_F1_Macro', 'Multiclass_ACC', 'Multiclass_F1_Macro']:
        display_df[col] = display_df[col].apply(lambda x: f"{x:.3f}" if pd.notna(x) else "N/A")

    print("\n" + display_df[['Strategy', 'Binary_ACC', 'Binary_F1_Macro', 'Binary_Best_Model',
                           'Multiclass_ACC', 'Multiclass_F1_Macro', 'Multiclass_Best_Model']].to_string(index=False))

    if save_dir:
        comparison_df.to_csv(os.path.join(save_dir, 'validation_strategy_comparison_full.csv'), index=False)
        print(f"\n✅ Saved full comparison table to: {save_dir}")

        thesis_table = comparison_df[['Strategy', 'Binary_ACC', 'Binary_F1_Macro',
                                      'Multiclass_ACC', 'Multiclass_F1_Macro']].copy()
        thesis_table.to_csv(os.path.join(save_dir, 'validation_strategy_comparison_thesis.csv'), index=False)
        print(f"✅ Saved thesis-ready comparison table to: {save_dir}")

        latex_table = create_latex_comparison_table(comparison_df)
        with open(os.path.join(save_dir, 'validation_strategy_comparison_latex.txt'), 'w') as f:
            f.write(latex_table)
        print(f"✅ Saved LaTeX table to: {save_dir}")

    print("\n" + "="*100)
    print("✅ VALIDATION COMPARISON TABLE COMPLETED")
    print("="*100)

    return comparison_df

def create_latex_comparison_table(comparison_df):
    """Create a LaTeX formatted table from the comparison results."""
    formatted_data = []
    for _, row in comparison_df.iterrows():
        binary_acc = f"{row['Binary_ACC']:.3f}" if pd.notna(row['Binary_ACC']) else "N/A"
        binary_f1 = f"{row['Binary_F1_Macro']:.3f}" if pd.notna(row['Binary_F1_Macro']) else "N/A"
        binary_model = row['Binary_Best_Model']
        multiclass_acc = f"{row['Multiclass_ACC']:.3f}" if pd.notna(row['Multiclass_ACC']) else "N/A"
        multiclass_f1 = f"{row['Multiclass_F1_Macro']:.3f}" if pd.notna(row['Multiclass_F1_Macro']) else "N/A"
        multiclass_model = row['Multiclass_Best_Model']

        formatted_data.append({
            'Strategy': row['Strategy'],
            'Binary_ACC': binary_acc,
            'Binary_F1': binary_f1,
            'Binary_Model': binary_model,
            'Multiclass_ACC': multiclass_acc,
            'Multiclass_F1': multiclass_f1,
            'Multiclass_Model': multiclass_model
        })

    formatted_df = pd.DataFrame(formatted_data)

    latex_lines = []
    latex_lines.append("\\begin{table}[htbp]")
    latex_lines.append("\\centering")
    latex_lines.append("\\caption{Validation Strategy Comparison: Binary (Tuned) vs Multiclass (Fixed CatBoost)}")
    latex_lines.append("\\label{tab:validation_comparison}")
    latex_lines.append("\\begin{tabular}{lcccccc}")
    latex_lines.append("\\hline")
    latex_lines.append("\\textbf{Strategy} & \\multicolumn{3}{c}{\\textbf{Binary (Rest vs Stress)}} & \\multicolumn{3}{c}{\\textbf{Multiclass (Rest, Highway, City)}} \\\\")
    latex_lines.append("\\cline{2-4} \\cline{5-7}")
    latex_lines.append(" & \\textbf{ACC} & \\textbf{F1} & \\textbf{Best Model} & \\textbf{ACC} & \\textbf{F1} & \\textbf{Best Model} \\\\")
    latex_lines.append("\\hline")

    for _, row in formatted_df.iterrows():
        latex_lines.append(f"{row['Strategy']} & {row['Binary_ACC']} & {row['Binary_F1']} & {row['Binary_Model']} & {row['Multiclass_ACC']} & {row['Multiclass_F1']} & {row['Multiclass_Model']} \\\\")

    latex_lines.append("\\hline")
    latex_lines.append("\\end{tabular}")
    latex_lines.append("\\end{table}")

    return "\n".join(latex_lines)

print("✅ Validation comparison functions loaded!")

# Final Execution - Run Complete Pipeline

In [ ]:
# ==========================================================
# CELL 17: FINAL EXECUTION
# ==========================================================

if __name__ == "__main__":
    import time
    start_time = time.time()

    print("\n" + "="*100)
    print("UNIFIED GSR STRESS CLASSIFICATION PIPELINE")
    print("Master's Thesis - Physiological Stress Recognition")
    print("="*100)
    print(f"\n Main results directory: {MAIN_RESULTS_DIR}")
    print(f"   Multiclass results: {MULTICLASS_DIR}")
    print(f"   Binary results: {BINARY_DIR}")
    print(f" Autorank approach: {AUTORANK_APPROACH}")
    print(f" Significance level: α = {AUTORANK_ALPHA}")
    print("\n PARAMETER STRATEGY:")
    print("   Binary: PRE-TUNED hyperparameters (ExtraTrees F1=0.830)")
    print("   Multiclass: FIXED CatBoost parameters (iterations=700, depth=6, lr=0.03) (F1=0.608)")
    print("\n   All hyperparameters were pre-tuned and are hard-coded.")
    print("   No hyperparameter tuning is performed during execution.")

    print("\n🔧 MODIFICATIONS INCLUDED:")
    print("    Raw + normalized confusion matrix saved")
    print("    TN, FP, FN, TP, Sensitivity, Specificity calculated")
    print("    Dynamic y-axis for driver performance plots")
    print("    Binary class distribution automatically saved")
    print("    Confusion matrix summary CSV generated")
    print("    Autorank with allow_insignificant=True")
    print("    Fixed: Handles both binary and multiclass confusion matrices")
    print("    FIXED: Binary and multiclass results saved in SEPARATE folders")
    print("    NEW: Validation comparison table generation")

    RUN_ABLATION = True

    all_results = {
        'binary': {},
        'multiclass': {}
    }

    # ==========================================================
    # Run Multiclass
    # ==========================================================
    print("\n" + "="*100)
    print("MULTICLASS CLASSIFICATION: Rest, Highway, City")
    print("="*100)
    print("🔧 Using FIXED CatBoost parameters (iterations=700, depth=6, lr=0.03)")
    print(f" Saving to: {MULTICLASS_DIR}")
    multiclass_results = run_unified_pipeline('multiclass', run_ablation=RUN_ABLATION, save_dir=MULTICLASS_DIR)
    all_results['multiclass'] = multiclass_results['fold_results']

    # ==========================================================
    # Run Binary: Rest vs. Stress
    # ==========================================================
    print("\n" + "="*100)
    print("BINARY CLASSIFICATION: Rest vs. Stress")
    print("="*100)
    print("🔧 Using PRE-TUNED hyperparameters (ExtraTrees F1=0.830)")
    print(f" Saving to: {BINARY_DIR}")
    binary_results = run_unified_pipeline('binary_stress', run_ablation=RUN_ABLATION, save_dir=BINARY_DIR)
    all_results['binary'] = binary_results['fold_results']

    # ==========================================================
    # Generate Validation Comparison Table
    # ==========================================================
    print("\n" + "="*100)
    print("GENERATING VALIDATION COMPARISON TABLE")
    print("="*100)

    comparison_df = generate_validation_comparison_table(
        results_dict=all_results,
        save_dir=MAIN_RESULTS_DIR
    )

    # ==========================================================
    # Final Summary
    # ==========================================================
    elapsed = time.time() - start_time
    print("\n" + "="*100)
    print(" ALL ANALYSES COMPLETED SUCCESSFULLY!")
    print("="*100)
    print(f"\n  Total runtime: {elapsed/60:.1f} minutes")
    print(f"\n Main results saved to: {MAIN_RESULTS_DIR}")
    print(f"   Multiclass: {MULTICLASS_DIR}")
    print(f"   Binary: {BINARY_DIR}")

    print("\n SUMMARY OF RESULTS:")
    print("  Binary (Pre-tuned): ExtraTrees F1=0.830, Accuracy=84.9%")
    print("  Multiclass (Fixed CatBoost): F1=0.608, Accuracy=66.0%")

    print("\n FILES GENERATED:")
    print("\n   Multiclass folder contains:")
    print("   • summary_LOSO.csv (multiclass)")
    print("   • summary_Group_5Fold_CV.csv (multiclass)")
    print("   • summary_Random_80_20.csv (multiclass)")
    print("   • confusion_matrix_summary_*.csv (multiclass - 3 classes)")
    print("   • main_results_multiclass.csv")
    print("   • fold_results_multiclass.csv")

    print("\n   Binary folder contains:")
    print("   • summary_LOSO.csv (binary)")
    print("   • summary_Group_5Fold_CV.csv (binary)")
    print("   • summary_Random_80_20.csv (binary)")
    print("   • confusion_matrix_summary_*.csv (binary - 2 classes)")
    print("   • main_results_binary_stress.csv")
    print("   • fold_results_binary_stress.csv")
    print("   • binary_class_distribution.csv")

    print("\n   Main folder contains:")
    print("   • validation_strategy_comparison_full.csv")
    print("   • validation_strategy_comparison_thesis.csv")
    print("   • validation_strategy_comparison_latex.txt")

    print("\n NO OVERWRITING - Both multiclass and binary results are preserved!")

    print("\n" + "="*100)
    print("FINAL VALIDATION COMPARISON TABLE")
    print("   Binary: PRE-TUNED | Multiclass: FIXED CatBoost")
    print("="*100)
    display_df = comparison_df.copy()
    for col in ['Binary_ACC', 'Binary_F1_Macro', 'Multiclass_ACC', 'Multiclass_F1_Macro']:
        display_df[col] = display_df[col].apply(lambda x: f"{x:.3f}" if pd.notna(x) else "N/A")
    print("\n" + display_df[['Strategy', 'Binary_ACC', 'Binary_F1_Macro', 'Binary_Best_Model',
                           'Multiclass_ACC', 'Multiclass_F1_Macro', 'Multiclass_Best_Model']].to_string(index=False))

    print("\n" + "="*100)
    print("PIPELINE COMPLETED SUCCESSFULLY!")
    print("="*100)